Toyota

In [3]:
"""
DriveArabia UAE Toyota Price Scraper
=====================================
Scrapes all Toyota models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/toyota/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_toyota.py
    python scrape_drivearabia_toyota.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_toyota.py --models toyota-land-cruiser toyota-camry
    python scrape_drivearabia_toyota.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_toyota.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
TOYOTA_INDEX = f"{BASE_URL}/carprices/uae/toyota/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|toyota)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": TOYOTA_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "toyota-4runner", "toyota-avalon", "toyota-avanza", "toyota-c-hr",
    "toyota-camry", "toyota-corolla", "toyota-corolla-cross", "toyota-fortuner",
    "toyota-granvia", "toyota-hiace", "toyota-hilux", "toyota-innova",
    "toyota-land-cruiser", "toyota-land-cruiser-70-series",
    "toyota-land-cruiser-prado", "toyota-prado", "toyota-rav4", "toyota-rush",
    "toyota-sequoia", "toyota-starlet", "toyota-tundra", "toyota-vios",
    "toyota-yaris", "toyota-yaris-cross",
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering Toyota models from %s", TOYOTA_INDEX)
    soup = fetch(session, TOYOTA_INDEX)
    if not soup:
        log.error("Failed to fetch Toyota index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/toyota/(toyota-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/toyota/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/drivearabia_toyota_prices.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_toyota import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["toyota-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape Toyota vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/drivearabia_toyota_prices.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



13:42:20 [INFO] Discovering Toyota models from https://www.drivearabia.com/carprices/uae/toyota/
13:42:20 [INFO] Discovered 39 models
13:42:20 [INFO] Year range: 2022â€“2026  |  Models: 39  |  Total pages: 195
13:42:22 [INFO] [1/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2022/
13:42:23 [INFO] [2/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2023/
13:42:24 [INFO] [3/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2024/
13:42:25 [INFO] [4/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2025/
13:42:26 [INFO] [5/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2026/
13:42:27 [INFO] [6/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-aurion/2022/
13:42:28 [INFO] [7/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-aurion/2023/
13:42:29 [INFO] [8/195] https://www.drivearabia.com/carprices/uae/toyota/toyota-aurion/2024/
13:42:30 [INFO] [9/195] https://www.drivearabia.com/carprices/uae/

KeyboardInterrupt: 

## Toyota

In [1]:
"""
DriveArabia UAE Toyota Price Scraper
=====================================
Scrapes all Toyota models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/toyota/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_toyota.py
    python scrape_drivearabia_toyota.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_toyota.py --models toyota-land-cruiser toyota-camry
    python scrape_drivearabia_toyota.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_toyota.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
TOYOTA_INDEX = f"{BASE_URL}/carprices/uae/toyota/"

DEFAULT_START_YEAR = 1995
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|toyota)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


def model_name_from_slug(model_slug: str) -> str:
    parts = [
        p
        for p in model_slug.replace("toyota-", "").replace("-", " ").split()
        if p
    ]
    return " ".join(parts)


def clean_trim_name(text: str, model_slug: str = "") -> str:
    """
    Keep only the trim label, not page labels, prices, links, or model prefixes.
    Examples:
      "Avalon Limited" -> "Limited"
      "2.5L I4 E FWDAED 109,900 - 110,000" -> "2.5L I4 E FWD"
      "Toyota Camry XLE" -> "XLE"
    """
    if not text:
        return ""

    trim = re.sub(r"\s+", " ", str(text)).strip()
    trim = re.split(r"AED", trim, maxsplit=1, flags=re.I)[0].strip()
    trim = re.sub(r"\b(Contact Dealer|See Similar Cars|Check Used Price)\b.*$", "", trim, flags=re.I).strip()
    trim = re.sub(r"^[\W_]+|[\W_]+$", "", trim)

    model_name = model_name_from_slug(model_slug)
    if model_name:
        # Remove "Toyota <model>" or "<model>" only when a trim remains.
        model_pattern = re.escape(model_name).replace(r"\ ", r"\s+")
        patterns = [
            rf"^toyota\s+{model_pattern}\s+(.+)$",
            rf"^{model_pattern}\s+(.+)$",
        ]
        for pattern in patterns:
            m = re.match(pattern, trim, re.I)
            if m and m.group(1).strip():
                trim = m.group(1).strip()
                break

    return re.sub(r"\s+", " ", trim).strip()


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": TOYOTA_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        trim_name = clean_trim_name(trim_name, model_slug)
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        trim_name = clean_trim_name(trim_name, model_slug)
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "toyota-86",
    "toyota-aurion",
    "toyota-avalon",
    "toyota-avanza",
    "toyota-c-hr",
    "toyota-camry",
    "toyota-corolla",
    "toyota-corolla-cross",
    "toyota-crown",
    "toyota-fj-cruiser",
    "toyota-fortuner",
    "toyota-gr-corolla",
    "toyota-gr-yaris",
    "toyota-gr86",
    "toyota-granvia",
    "toyota-hiace",
    "toyota-highlander",
    "toyota-hilux",
    "toyota-innova",
    "toyota-land-cruiser",
    "toyota-land-cruiser-70",
    "toyota-land-cruiser-gr-sport",
    "toyota-land-cruiser-pickup",
    "toyota-land-cruiser-prado",
    "toyota-land-cruiser-prado-swb",
    "toyota-liteace",
    "toyota-previa",
    "toyota-prius",
    "toyota-rav4",
    "toyota-raize",
    "toyota-rush",
    "toyota-sequoia",
    "toyota-supra",
    "toyota-urban-cruiser",
    "toyota-veloz",
    "toyota-yaris",
    "toyota-yaris-sedan",
    "toyota-zelas",
    "toyota-xa"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering Toyota models from %s", TOYOTA_INDEX)
    soup = fetch(session, TOYOTA_INDEX)
    if not soup:
        log.error("Failed to fetch Toyota index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/toyota/(toyota-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/toyota/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/drivearabia_toyota_prices_newest_final.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_toyota import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["toyota-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape Toyota vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/drivearabia_toyota_prices_newest_final.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



11:01:39 [INFO] Discovering Toyota models from https://www.drivearabia.com/carprices/uae/toyota/
11:01:40 [INFO] Discovered 39 models
11:01:40 [INFO] Year range: 1995â€“2026  |  Models: 39  |  Total pages: 1248
11:01:43 [INFO] [1/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/1995/
11:01:45 [INFO] [2/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/1996/
11:01:46 [INFO] [3/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/1997/
11:01:47 [INFO] [4/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/1998/
11:01:48 [INFO] [5/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/1999/
11:01:49 [INFO] [6/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2000/
11:01:51 [INFO] [7/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2001/
11:01:52 [INFO] [8/1248] https://www.drivearabia.com/carprices/uae/toyota/toyota-86/2002/
11:01:53 [INFO] [9/1248] https://www.drivearabia.com/carprices/uae/to

KeyboardInterrupt: 

## Nissan

In [6]:
"""
DriveArabia UAE nissan Price Scraper
=====================================
Scrapes all nissan models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/nissan/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_nissan.py
    python scrape_drivearabia_nissan.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_nissan.py --models nissan-land-cruiser nissan-camry
    python scrape_drivearabia_nissan.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_nissan.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
nissan_INDEX = f"{BASE_URL}/carprices/uae/nissan/"

DEFAULT_START_YEAR = 1995
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|nissan)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


def model_name_from_slug(model_slug: str) -> str:
    parts = [
        p
        for p in model_slug.replace("nissan-", "").replace("-", " ").split()
        if p
    ]
    return " ".join(parts)


def clean_trim_name(text: str, model_slug: str = "") -> str:
    """
    Keep only the trim label, not page labels, prices, links, or model prefixes.
    Examples:
      "Avalon Limited" -> "Limited"
      "2.5L I4 E FWDAED 109,900 - 110,000" -> "2.5L I4 E FWD"
      "nissan Camry XLE" -> "XLE"
    """
    if not text:
        return ""

    trim = re.sub(r"\s+", " ", str(text)).strip()
    trim = re.split(r"AED", trim, maxsplit=1, flags=re.I)[0].strip()
    trim = re.sub(r"\b(Contact Dealer|See Similar Cars|Check Used Price)\b.*$", "", trim, flags=re.I).strip()
    trim = re.sub(r"^[\W_]+|[\W_]+$", "", trim)

    model_name = model_name_from_slug(model_slug)
    if model_name:
        # Remove "nissan <model>" or "<model>" only when a trim remains.
        model_pattern = re.escape(model_name).replace(r"\ ", r"\s+")
        patterns = [
            rf"^nissan\s+{model_pattern}\s+(.+)$",
            rf"^{model_pattern}\s+(.+)$",
        ]
        for pattern in patterns:
            m = re.match(pattern, trim, re.I)
            if m and m.group(1).strip():
                trim = m.group(1).strip()
                break

    return re.sub(r"\s+", " ", trim).strip()


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": nissan_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        trim_name = clean_trim_name(trim_name, model_slug)
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        trim_name = clean_trim_name(trim_name, model_slug)
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
 "nissan-350z",
"nissan-370z",
"nissan-370z-roadster",
"nissan-altima",
"nissan-altima-coupe",
"nissan-ariya",
"nissan-armada",
"nissan-gt-r",
"nissan-juke",
"nissan-kicks",
"nissan-magnite",
"nissan-maxima",
"nissan-micra",
"nissan-murano",
"nissan-navara",
"nissan-pathfinder",
"nissan-pathfinder-classic",
"nissan-patrol",
"nissan-patrol-nismo",
"nissan-patrol-pro-4x",
"nissan-patrol-pickup",
"nissan-patrol-safari",
"nissan-pickup",
"nissan-qashqai",
"nissan-sentra",
"nissan-sunny",
"nissan-tekna",
"nissan-tiida",
"nissan-urvan",
"nissan-x-trail",
"nissan-xterra",
"nissan-z",
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering nissan models from %s", nissan_INDEX)
    soup = fetch(session, nissan_INDEX)
    if not soup:
        log.error("Failed to fetch nissan index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/nissan/(nissan-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/nissan/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/nissan.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_nissan import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["nissan-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape nissan vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/nissan.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:41:48 [INFO] Discovering nissan models from https://www.drivearabia.com/carprices/uae/nissan/
15:41:48 [INFO] Discovered 32 models
15:41:48 [INFO] Year range: 1995â€“2026  |  Models: 32  |  Total pages: 1024
15:41:50 [INFO] [1/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/1995/
15:41:51 [INFO] [2/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/1996/
15:41:52 [INFO] [3/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/1997/
15:41:53 [INFO] [4/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/1998/
15:41:54 [INFO] [5/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/1999/
15:41:55 [INFO] [6/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/2000/
15:41:56 [INFO] [7/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/2001/
15:41:58 [INFO] [8/1024] https://www.drivearabia.com/carprices/uae/nissan/nissan-350z/2002/
15:41:59 [INFO] [9/1024] https://www.drivearabia.com/

In [ ]:
"""
DriveArabia UAE hyundai Price Scraper
=====================================
Scrapes all hyundai models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/hyundai/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_hyundai.py
    python scrape_drivearabia_hyundai.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_hyundai.py --models hyundai-land-cruiser hyundai-camry
    python scrape_drivearabia_hyundai.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_hyundai.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
hyundai_INDEX = f"{BASE_URL}/carprices/uae/hyundai/"

DEFAULT_START_YEAR = 1995
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|hyundai|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|hyundai)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


def model_name_from_slug(model_slug: str) -> str:
    parts = [
        p
        for p in model_slug.replace("hyundai-", "").replace("-", " ").split()
        if p
    ]
    return " ".join(parts)


def clean_trim_name(text: str, model_slug: str = "") -> str:
    """
    Keep only the trim label, not page labels, prices, links, or model prefixes.
    Examples:
      "Avalon Limited" -> "Limited"
      "2.5L I4 E FWDAED 109,900 - 110,000" -> "2.5L I4 E FWD"
      "hyundai Camry XLE" -> "XLE"
    """
    if not text:
        return ""

    trim = re.sub(r"\s+", " ", str(text)).strip()
    trim = re.split(r"AED", trim, maxsplit=1, flags=re.I)[0].strip()
    trim = re.sub(r"\b(Contact Dealer|See Similar Cars|Check Used Price)\b.*$", "", trim, flags=re.I).strip()
    trim = re.sub(r"^[\W_]+|[\W_]+$", "", trim)

    model_name = model_name_from_slug(model_slug)
    if model_name:
        # Remove "hyundai <model>" or "<model>" only when a trim remains.
        model_pattern = re.escape(model_name).replace(r"\ ", r"\s+")
        patterns = [
            rf"^hyundai\s+{model_pattern}\s+(.+)$",
            rf"^{model_pattern}\s+(.+)$",
        ]
        for pattern in patterns:
            m = re.match(pattern, trim, re.I)
            if m and m.group(1).strip():
                trim = m.group(1).strip()
                break

    return re.sub(r"\s+", " ", trim).strip()


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": hyundai_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        trim_name = clean_trim_name(trim_name, model_slug)
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        trim_name = clean_trim_name(trim_name, model_slug)
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
"hyundai-accent",
"hyundai-atos-prime",
"hyundai-azera",
"hyundai-centennial",
"hyundai-coupe",
"hyundai-creta",
"hyundai-elantra",
"hyundai-elantra-coupe",
"hyundai-genesis",
"hyundai-genesis-coupe",
"hyundai-getz",
"hyundai-grand-santa-fe",
"hyundai-grand-i10",
"hyundai-h1",
"hyundai-ioniq-5",
"hyundai-kona",
"hyundai-kona-hybrid",
"hyundai-matrix",
"hyundai-palisade",
"hyundai-santa-fe",
"hyundai-sonata",
"hyundai-stargazer",
"hyundai-staria",
"hyundai-terracan",
"hyundai-trajet",
"hyundai-tucson",
"hyundai-veloster",
"hyundai-veloster-n",
"hyundai-veloster-turbo",
"hyundai-venue",
"hyundai-veracruz",
"hyundai-i10",
"hyundai-i20",
"hyundai-i30",
"hyundai-i40",
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering hyundai models from %s", hyundai_INDEX)
    soup = fetch(session, hyundai_INDEX)
    if not soup:
        log.error("Failed to fetch hyundai index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/hyundai/(hyundai-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/hyundai/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/hyundai.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_hyundai import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["hyundai-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape hyundai vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/hyundai.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:09:22 [INFO] Discovering hyundai models from https://www.drivearabia.com/carprices/uae/hyundai/
16:09:22 [INFO] Discovered 35 models
16:09:22 [INFO] Year range: 1995â€“2026  |  Models: 35  |  Total pages: 1120
16:09:25 [INFO] [1/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/1995/
16:09:27 [INFO] [2/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/1996/
16:09:28 [INFO] [3/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/1997/
16:09:29 [INFO] [4/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/1998/
16:09:30 [INFO] [5/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/1999/
16:09:32 [INFO] [6/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/2000/
16:09:33 [INFO] [7/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/2001/
16:09:34 [INFO] [8/1120] https://www.drivearabia.com/carprices/uae/hyundai/hyundai-accent/2002/
16:09:35 [INFO] [9/

In [3]:
"""
DriveArabia UAE chevrolet Price Scraper
=====================================
Scrapes all chevrolet models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/chevrolet/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_chevrolet.py
    python scrape_drivearabia_chevrolet.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_chevrolet.py --models chevrolet-land-cruiser chevrolet-camry
    python scrape_drivearabia_chevrolet.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_chevrolet.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
chevrolet_INDEX = f"{BASE_URL}/carprices/uae/chevrolet/"

DEFAULT_START_YEAR = 1995
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(chevrolet|honda|chevrolet|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|chevrolet)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


def model_name_from_slug(model_slug: str) -> str:
    parts = [
        p
        for p in model_slug.replace("chevrolet-", "").replace("-", " ").split()
        if p
    ]
    return " ".join(parts)


def clean_trim_name(text: str, model_slug: str = "") -> str:
    """
    Keep only the trim label, not page labels, prices, links, or model prefixes.
    Examples:
      "Avalon Limited" -> "Limited"
      "2.5L I4 E FWDAED 109,900 - 110,000" -> "2.5L I4 E FWD"
      "chevrolet Camry XLE" -> "XLE"
    """
    if not text:
        return ""

    trim = re.sub(r"\s+", " ", str(text)).strip()
    trim = re.split(r"AED", trim, maxsplit=1, flags=re.I)[0].strip()
    trim = re.sub(r"\b(Contact Dealer|See Similar Cars|Check Used Price)\b.*$", "", trim, flags=re.I).strip()
    trim = re.sub(r"^[\W_]+|[\W_]+$", "", trim)

    model_name = model_name_from_slug(model_slug)
    if model_name:
        # Remove "chevrolet <model>" or "<model>" only when a trim remains.
        model_pattern = re.escape(model_name).replace(r"\ ", r"\s+")
        patterns = [
            rf"^chevrolet\s+{model_pattern}\s+(.+)$",
            rf"^{model_pattern}\s+(.+)$",
        ]
        for pattern in patterns:
            m = re.match(pattern, trim, re.I)
            if m and m.group(1).strip():
                trim = m.group(1).strip()
                break

    return re.sub(r"\s+", " ", trim).strip()


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": chevrolet_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        trim_name = clean_trim_name(trim_name, model_slug)
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        trim_name = clean_trim_name(trim_name, model_slug)
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "chevrolet-avalanche",
    "chevrolet-aveo",
    "chevrolet-aveo-hatchback",
    "chevrolet-aveo5",
    "chevrolet-blazer",
    "chevrolet-bolt",
    "chevrolet-bolt-euv",
    "chevrolet-csv-cr8",
    "chevrolet-camaro",
    "chevrolet-camaro-convertible",
    "chevrolet-camaro-zl1",
    "chevrolet-caprice",
    "chevrolet-captiva",
    "chevrolet-captiva-ev",
    "chevrolet-captiva-phev",
    "chevrolet-colorado",
    "chevrolet-corvette",
    "chevrolet-cruze",
    "chevrolet-cruze-hatchback",
    "chevrolet-epica",
    "chevrolet-equinox",
    "chevrolet-equinox-ev",
    "chevrolet-express",
    "chevrolet-groove",
    "chevrolet-impala",
    "chevrolet-lumina",
    "chevrolet-malibu",
    "chevrolet-optra",
    "chevrolet-silverado",
    "chevrolet-sonic",
    "chevrolet-sonic-hatchback",
    "chevrolet-spark",
    "chevrolet-spark-euv",
    "chevrolet-suburban",
    "chevrolet-t-series",
    "chevrolet-tahoe",
    "chevrolet-trailblazer",
    "chevrolet-traverse",
    "chevrolet-trax",
    "chevrolet-uplander",
]

def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering chevrolet models from %s", chevrolet_INDEX)
    soup = fetch(session, chevrolet_INDEX)
    if not soup:
        log.error("Failed to fetch chevrolet index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/chevrolet/(chevrolet-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/chevrolet/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/chevrolet.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_chevrolet import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["chevrolet-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape chevrolet vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/chevrolet.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



13:11:15 [INFO] Discovering chevrolet models from https://www.drivearabia.com/carprices/uae/chevrolet/
13:11:16 [INFO] Discovered 40 models
13:11:16 [INFO] Year range: 1995â€“2026  |  Models: 40  |  Total pages: 1280
13:11:18 [INFO] [1/1280] https://www.drivearabia.com/carprices/uae/chevrolet/chevrolet-avalanche/1995/
13:11:19 [INFO] [2/1280] https://www.drivearabia.com/carprices/uae/chevrolet/chevrolet-avalanche/1996/
13:11:20 [INFO] [3/1280] https://www.drivearabia.com/carprices/uae/chevrolet/chevrolet-avalanche/1997/
13:11:21 [INFO] [4/1280] https://www.drivearabia.com/carprices/uae/chevrolet/chevrolet-avalanche/1998/
13:11:22 [INFO] [5/1280] https://www.drivearabia.com/carprices/uae/chevrolet/chevrolet-avalanche/1999/
13:11:23 [INFO] [6/1280] https://www.drivearabia.com/carprices/uae/chevrolet/chevrolet-avalanche/2000/
13:11:24 [INFO] [7/1280] https://www.drivearabia.com/carprices/uae/chevrolet/chevrolet-avalanche/2001/
13:11:26 [INFO] [8/1280] https://www.drivearabia.com/carprices

In [4]:
"""
DriveArabia UAE ford Price Scraper
=====================================
Scrapes all ford models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/ford/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_ford.py
    python scrape_drivearabia_ford.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_ford.py --models ford-land-cruiser ford-camry
    python scrape_drivearabia_ford.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_ford.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
ford_INDEX = f"{BASE_URL}/carprices/uae/ford/"

DEFAULT_START_YEAR = 1995
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(ford|honda|ford|ford|ford|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|ford)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


def model_name_from_slug(model_slug: str) -> str:
    parts = [
        p
        for p in model_slug.replace("ford-", "").replace("-", " ").split()
        if p
    ]
    return " ".join(parts)


def clean_trim_name(text: str, model_slug: str = "") -> str:
    """
    Keep only the trim label, not page labels, prices, links, or model prefixes.
    Examples:
      "Avalon Limited" -> "Limited"
      "2.5L I4 E FWDAED 109,900 - 110,000" -> "2.5L I4 E FWD"
      "ford Camry XLE" -> "XLE"
    """
    if not text:
        return ""

    trim = re.sub(r"\s+", " ", str(text)).strip()
    trim = re.split(r"AED", trim, maxsplit=1, flags=re.I)[0].strip()
    trim = re.sub(r"\b(Contact Dealer|See Similar Cars|Check Used Price)\b.*$", "", trim, flags=re.I).strip()
    trim = re.sub(r"^[\W_]+|[\W_]+$", "", trim)

    model_name = model_name_from_slug(model_slug)
    if model_name:
        # Remove "ford <model>" or "<model>" only when a trim remains.
        model_pattern = re.escape(model_name).replace(r"\ ", r"\s+")
        patterns = [
            rf"^ford\s+{model_pattern}\s+(.+)$",
            rf"^{model_pattern}\s+(.+)$",
        ]
        for pattern in patterns:
            m = re.match(pattern, trim, re.I)
            if m and m.group(1).strip():
                trim = m.group(1).strip()
                break

    return re.sub(r"\s+", " ", trim).strip()


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": ford_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        trim_name = clean_trim_name(trim_name, model_slug)
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        trim_name = clean_trim_name(trim_name, model_slug)
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
  "ford-bronco",
    "ford-bronco-2-door",
    "ford-bronco-raptor",
    "ford-crown-victoria",
    "ford-ecosport",
    "ford-edge",
    "ford-escape",
    "ford-escort",
    "ford-everest",
    "ford-expedition",
    "ford-explorer",
    "ford-f-150",
    "ford-f-150-raptor",
    "ford-fiesta",
    "ford-figo",
    "ford-five-hundred",
    "ford-flex",
    "ford-focus",
    "ford-focus-st",
    "ford-freestar",
    "ford-fusion",
    "ford-gt",
    "ford-mondeo",
    "ford-mustang",
    "ford-mustang-convertible",
    "ford-mustang-mach-1",
    "ford-mustang-mach-e",
    "ford-ranger",
    "ford-ranger-raptor",
    "ford-shelby-gt500",
    "ford-taurus",
    "ford-territory",
    "ford-tourneo-custom",
    "ford-transit",
    "ford-transit-custom",
]

def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering ford models from %s", ford_INDEX)
    soup = fetch(session, ford_INDEX)
    if not soup:
        log.error("Failed to fetch ford index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/ford/(ford-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/ford/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/ford.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_ford import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["ford-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape ford vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/ford.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



13:40:02 [INFO] Discovering ford models from https://www.drivearabia.com/carprices/uae/ford/
13:40:02 [INFO] Discovered 35 models
13:40:02 [INFO] Year range: 1995â€“2026  |  Models: 35  |  Total pages: 1120
13:40:04 [INFO] [1/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/1995/
13:40:05 [INFO] [2/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/1996/
13:40:06 [INFO] [3/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/1997/
13:40:07 [INFO] [4/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/1998/
13:40:08 [INFO] [5/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/1999/
13:40:09 [INFO] [6/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/2000/
13:40:11 [INFO] [7/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/2001/
13:40:12 [INFO] [8/1120] https://www.drivearabia.com/carprices/uae/ford/ford-bronco/2002/
13:40:13 [INFO] [9/1120] https://www.drivearabia.com/carprices/uae/ford/f

In [6]:
"""
DriveArabia UAE honda Price Scraper
=====================================
Scrapes all honda models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/honda/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_honda.py
    python scrape_drivearabia_honda.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_honda.py --models honda-land-cruiser honda-camry
    python scrape_drivearabia_honda.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_honda.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
honda_INDEX = f"{BASE_URL}/carprices/uae/honda/"

DEFAULT_START_YEAR = 1995
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(honda|honda|honda|ford|honda|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|honda)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


def model_name_from_slug(model_slug: str) -> str:
    parts = [
        p
        for p in model_slug.replace("honda-", "").replace("-", " ").split()
        if p
    ]
    return " ".join(parts)


def clean_trim_name(text: str, model_slug: str = "") -> str:
    """
    Keep only the trim label, not page labels, prices, links, or model prefixes.
    Examples:
      "Avalon Limited" -> "Limited"
      "2.5L I4 E FWDAED 109,900 - 110,000" -> "2.5L I4 E FWD"
      "honda Camry XLE" -> "XLE"
    """
    if not text:
        return ""

    trim = re.sub(r"\s+", " ", str(text)).strip()
    trim = re.split(r"AED", trim, maxsplit=1, flags=re.I)[0].strip()
    trim = re.sub(r"\b(Contact Dealer|See Similar Cars|Check Used Price)\b.*$", "", trim, flags=re.I).strip()
    trim = re.sub(r"^[\W_]+|[\W_]+$", "", trim)

    model_name = model_name_from_slug(model_slug)
    if model_name:
        # Remove "honda <model>" or "<model>" only when a trim remains.
        model_pattern = re.escape(model_name).replace(r"\ ", r"\s+")
        patterns = [
            rf"^honda\s+{model_pattern}\s+(.+)$",
            rf"^{model_pattern}\s+(.+)$",
        ]
        for pattern in patterns:
            m = re.match(pattern, trim, re.I)
            if m and m.group(1).strip():
                trim = m.group(1).strip()
                break

    return re.sub(r"\s+", " ", trim).strip()


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": honda_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        trim_name = clean_trim_name(trim_name, model_slug)
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        trim_name = clean_trim_name(trim_name, model_slug)
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
   "honda-accord",
    "honda-accord-coupe",
    "honda-br-v",
    "honda-city",
    "honda-civic",
    "honda-civic-type-r",
    "honda-cr-v",
    "honda-cr-z",
    "honda-crosstour",
    "honda-e",
    "honda-fit",
    "honda-hr-v",
    "honda-jazz",
    "honda-legend",
    "honda-odyssey",
    "honda-passport",
    "honda-pilot",
    "honda-ridgeline",
    "honda-s2000",
    "honda-stepwgn",
    "honda-stream",
    "honda-vezel",
    "honda-zr-v",]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering honda models from %s", honda_INDEX)
    soup = fetch(session, honda_INDEX)
    if not soup:
        log.error("Failed to fetch honda index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/honda/(honda-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/honda/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/honda.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_honda import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["honda-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape honda vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/honda.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



14:50:08 [INFO] Discovering honda models from https://www.drivearabia.com/carprices/uae/honda/
14:50:08 [INFO] Discovered 19 models
14:50:08 [INFO] Year range: 1995â€“2026  |  Models: 19  |  Total pages: 608
14:50:10 [INFO] [1/608] https://www.drivearabia.com/carprices/uae/honda/honda-accord/1995/
14:50:12 [INFO] [2/608] https://www.drivearabia.com/carprices/uae/honda/honda-accord/1996/
14:50:13 [INFO] [3/608] https://www.drivearabia.com/carprices/uae/honda/honda-accord/1997/
14:50:14 [INFO] [4/608] https://www.drivearabia.com/carprices/uae/honda/honda-accord/1998/
14:50:14 [WARNING]   â†’ 0 trims parsed â€” try --debug-html to inspect page structure
14:50:16 [INFO] [5/608] https://www.drivearabia.com/carprices/uae/honda/honda-accord/1999/
14:50:16 [WARNING]   â†’ 0 trims parsed â€” try --debug-html to inspect page structure
14:50:18 [INFO] [6/608] https://www.drivearabia.com/carprices/uae/honda/honda-accord/2000/
14:50:18 [WARNING]   â†’ 0 trims parsed â€” try --debug-html to inspect 

In [1]:
"""
DriveArabia UAE kia Price Scraper
=====================================
Scrapes all kia models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/kia/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_kia.py
    python scrape_drivearabia_kia.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_kia.py --models kia-land-cruiser kia-camry
    python scrape_drivearabia_kia.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_kia.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
kia_INDEX = f"{BASE_URL}/carprices/uae/kia/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|kia)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": kia_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "kia-cadenza",
    "kia-carens",
    "kia-carnival",
    "kia-cerato",
    "kia-cerato-hatchback",
    "kia-cerato-koup",
    "kia-ev5",
    "kia-ev6",
    "kia-ev9",
    "kia-k3",
    "kia-k4",
    "kia-k5",
    "kia-k8",
    "kia-k900",
    "kia-mohave",
    "kia-niro",
    "kia-opirus",
    "kia-optima",
    "kia-pegas",
    "kia-picanto",
    "kia-quoris",
    "kia-rio",
    "kia-rio-hatchback",
    "kia-seltos",
    "kia-sonet",
    "kia-sorento",
    "kia-soul",
    "kia-sportage",
    "kia-stinger",
    "kia-tasman",
    "kia-telluride"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering kia models from %s", kia_INDEX)
    soup = fetch(session, kia_INDEX)
    if not soup:
        log.error("Failed to fetch kia index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/kia/(kia-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/kia/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/drivearabia_kia_prices.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_kia import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["kia-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape kia vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/drivearabia_kia_prices.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



14:53:07 [INFO] Discovering kia models from https://www.drivearabia.com/carprices/uae/kia/
14:53:08 [INFO] Discovered 31 models
14:53:08 [INFO] Year range: 2022â€“2026  |  Models: 31  |  Total pages: 155
14:53:10 [INFO] [1/155] https://www.drivearabia.com/carprices/uae/kia/kia-cadenza/2022/
14:53:11 [INFO] [2/155] https://www.drivearabia.com/carprices/uae/kia/kia-cadenza/2023/
14:53:12 [INFO] [3/155] https://www.drivearabia.com/carprices/uae/kia/kia-cadenza/2024/
14:53:13 [INFO] [4/155] https://www.drivearabia.com/carprices/uae/kia/kia-cadenza/2025/
14:53:14 [INFO] [5/155] https://www.drivearabia.com/carprices/uae/kia/kia-cadenza/2026/
14:53:16 [INFO] [6/155] https://www.drivearabia.com/carprices/uae/kia/kia-carens/2022/
14:53:17 [INFO] [7/155] https://www.drivearabia.com/carprices/uae/kia/kia-carens/2023/
14:53:18 [INFO] [8/155] https://www.drivearabia.com/carprices/uae/kia/kia-carens/2024/
14:53:18 [INFO]   â†’ 3 trims
14:53:20 [INFO] [9/155] https://www.drivearabia.com/carprices/uae

In [2]:
"""
DriveArabia UAE mazda Price Scraper
=====================================
Scrapes all mazda models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/mazda/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_mazda.py
    python scrape_drivearabia_mazda.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_mazda.py --models mazda-land-cruiser mazda-camry
    python scrape_drivearabia_mazda.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_mazda.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
mazda_INDEX = f"{BASE_URL}/carprices/uae/mazda/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|mazda)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": mazda_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "mazda-4runner", "mazda-avalon", "mazda-avanza", "mazda-c-hr",
    "mazda-camry", "mazda-corolla", "mazda-corolla-cross", "mazda-fortuner",
    "mazda-granvia", "mazda-hiace", "mazda-hilux", "mazda-innova",
    "mazda-land-cruiser", "mazda-land-cruiser-70-series",
    "mazda-land-cruiser-prado", "mazda-prado", "mazda-rav4", "mazda-rush",
    "mazda-sequoia", "mazda-starlet", "mazda-tundra", "mazda-vios",
    "mazda-yaris", "mazda-yaris-cross",
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering mazda models from %s", mazda_INDEX)
    soup = fetch(session, mazda_INDEX)
    if not soup:
        log.error("Failed to fetch mazda index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/mazda/(mazda-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/mazda/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/mazda.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_mazda import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["mazda-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape mazda vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/mazda.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



14:57:16 [INFO] Discovering mazda models from https://www.drivearabia.com/carprices/uae/mazda/
14:57:19 [INFO] Discovered 17 models
14:57:19 [INFO] Year range: 2022â€“2026  |  Models: 17  |  Total pages: 85
14:57:23 [INFO] [1/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2/2022/
14:57:24 [INFO]   â†’ 3 trims
14:57:26 [INFO] [2/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2/2023/
14:57:27 [INFO] [3/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2/2024/
14:57:28 [INFO] [4/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2/2025/
14:57:29 [INFO] [5/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2/2026/
14:57:30 [INFO] [6/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2-sedan/2022/
14:57:31 [INFO] [7/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2-sedan/2023/
14:57:34 [INFO] [8/85] https://www.drivearabia.com/carprices/uae/mazda/mazda-2-sedan/2024/
14:57:35 [INFO] [9/85] https://www.drivearabia.com/carprices/uae/

In [3]:
"""
DriveArabia UAE gmc Price Scraper
=====================================
Scrapes all gmc models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/gmc/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_gmc.py
    python scrape_drivearabia_gmc.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_gmc.py --models gmc-land-cruiser gmc-camry
    python scrape_drivearabia_gmc.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_gmc.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
gmc_INDEX = f"{BASE_URL}/carprices/uae/gmc/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|gmc)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": gmc_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "mazda-2",
    "mazda-2-sedan",
    "mazda-3",
    "mazda-3-sedan",
    "mazda-6",
    "mazda-6-ultra",
    "mazda-6-wagon",
    "mazda-b-series",
    "mazda-cx-3",
    "mazda-cx-30",
    "mazda-cx-5",
    "mazda-cx-60",
    "mazda-cx-7",
    "mazda-cx-9",
    "mazda-cx-90",
    "mazda-mx-5",
    "mazda-mx-5-rf"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering gmc models from %s", gmc_INDEX)
    soup = fetch(session, gmc_INDEX)
    if not soup:
        log.error("Failed to fetch gmc index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/gmc/(gmc-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/gmc/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/gmc.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_gmc import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["gmc-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape gmc vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/gmc.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:01:00 [INFO] Discovering gmc models from https://www.drivearabia.com/carprices/uae/gmc/
15:01:00 [INFO] Discovered 9 models
15:01:00 [INFO] Year range: 2022â€“2026  |  Models: 9  |  Total pages: 45
15:01:04 [INFO] [1/45] https://www.drivearabia.com/carprices/uae/gmc/gmc-acadia/2022/
15:01:04 [INFO]   â†’ 4 trims
15:01:06 [INFO] [2/45] https://www.drivearabia.com/carprices/uae/gmc/gmc-acadia/2023/
15:01:06 [INFO]   â†’ 4 trims
15:01:08 [INFO] [3/45] https://www.drivearabia.com/carprices/uae/gmc/gmc-acadia/2024/
15:01:08 [INFO]   â†’ 4 trims
15:01:10 [INFO] [4/45] https://www.drivearabia.com/carprices/uae/gmc/gmc-acadia/2025/
15:01:10 [INFO]   â†’ 2 trims
15:01:12 [INFO] [5/45] https://www.drivearabia.com/carprices/uae/gmc/gmc-acadia/2026/
15:01:12 [INFO]   â†’ 2 trims
15:01:13 [INFO] [6/45] https://www.drivearabia.com/carprices/uae/gmc/gmc-canyon/2022/
15:01:14 [INFO] [7/45] https://www.drivearabia.com/carprices/uae/gmc/gmc-canyon/2023/
15:01:16 [INFO] [8/45] https://www.drivearabia.

In [7]:
"""
DriveArabia UAE mini Price Scraper
=====================================
Scrapes all mini models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/mini/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python mini.py
    python mini.py --start-year 2020 --end-year 2026
    python mini.py --models mini-land-cruiser mini-camry
    python mini.py --delay 2.0 --output my_prices.csv
    python mini.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
mini = f"{BASE_URL}/carprices/uae/mini/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"mini|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|mini)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": mini})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "mini-clubman",
    "mini-cooper",
    "mini-cooper-5-door",
    "mini-countryman",
    "mini-countryman-se",
    "mini-coupe",
    "mini-paceman",
    "mini-roadster"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering mini models from %s", mini)
    soup = fetch(session, mini)
    if not soup:
        log.error("Failed to fetch mini index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/mini/(mini-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/mini/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/mini.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from mini import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["mini-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape mini vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/mini.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:08:19 [INFO] Discovering mini models from https://www.drivearabia.com/carprices/uae/mini/
15:08:20 [INFO] Discovered 8 models
15:08:20 [INFO] Year range: 2022â€“2026  |  Models: 8  |  Total pages: 40
15:08:22 [INFO] [1/40] https://www.drivearabia.com/carprices/uae/mini/mini-clubman/2022/
15:08:22 [INFO]   â†’ 2 trims
15:08:24 [INFO] [2/40] https://www.drivearabia.com/carprices/uae/mini/mini-clubman/2023/
15:08:24 [INFO]   â†’ 2 trims
15:08:26 [INFO] [3/40] https://www.drivearabia.com/carprices/uae/mini/mini-clubman/2024/
15:08:27 [INFO] [4/40] https://www.drivearabia.com/carprices/uae/mini/mini-clubman/2025/
15:08:28 [INFO] [5/40] https://www.drivearabia.com/carprices/uae/mini/mini-clubman/2026/
15:08:29 [INFO] [6/40] https://www.drivearabia.com/carprices/uae/mini/mini-cooper/2022/
15:08:30 [INFO]   â†’ 4 trims
15:08:31 [INFO] [7/40] https://www.drivearabia.com/carprices/uae/mini/mini-cooper/2023/
15:08:31 [INFO]   â†’ 4 trims
15:08:33 [INFO] [8/40] https://www.drivearabia.com/carpr

In [8]:
"""
DriveArabia UAE byd Price Scraper
=====================================
Scrapes all byd models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/byd/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_byd.py
    python scrape_drivearabia_byd.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_byd.py --models byd-land-cruiser byd-camry
    python scrape_drivearabia_byd.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_byd.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
byd_INDEX = f"{BASE_URL}/carprices/uae/byd/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"byd|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|byd)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": byd_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "byd-atto-3",
    "byd-atto-8",
    "byd-f3",
    "byd-f5",
    "byd-f6",
    "byd-f7",
    "byd-han",
    "byd-qin-plus",
    "byd-qin-pro",
    "byd-s6",
    "byd-seal",
    "byd-seal-6",
    "byd-seal-7",
    "byd-sealion-5",
    "byd-sealion-7",
    "byd-shark-6",
    "byd-song-plus",
    "byd-song-pro",
    "byd-ti-7"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering byd models from %s", byd_INDEX)
    soup = fetch(session, byd_INDEX)
    if not soup:
        log.error("Failed to fetch byd index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/byd/(byd-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/byd/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/byd.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_byd import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["byd-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape byd vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/byd.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:11:17 [INFO] Discovering byd models from https://www.drivearabia.com/carprices/uae/byd/
15:11:17 [INFO] Discovered 19 models
15:11:17 [INFO] Year range: 2022â€“2026  |  Models: 19  |  Total pages: 95
15:11:19 [INFO] [1/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-3/2022/
15:11:21 [INFO] [2/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-3/2023/
15:11:22 [INFO] [3/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-3/2024/
15:11:22 [INFO]   â†’ 1 trims
15:11:24 [INFO] [4/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-3/2025/
15:11:24 [INFO]   â†’ 1 trims
15:11:25 [INFO] [5/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-3/2026/
15:11:26 [INFO]   â†’ 1 trims
15:11:27 [INFO] [6/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-8/2022/
15:11:28 [INFO] [7/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-8/2023/
15:11:30 [INFO] [8/95] https://www.drivearabia.com/carprices/uae/byd/byd-atto-8/2024/
15:11:31 [INFO] [9/

In [9]:
"""
DriveArabia UAE infiniti Price Scraper
=====================================
Scrapes all infiniti models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/infiniti/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_infiniti.py
    python scrape_drivearabia_infiniti.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_infiniti.py --models infiniti-land-cruiser infiniti-camry
    python scrape_drivearabia_infiniti.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_infiniti.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
infiniti_INDEX = f"{BASE_URL}/carprices/uae/infiniti/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"infiniti|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|infiniti)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": infiniti_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "infiniti-ex",
    "infiniti-fx",
    "infiniti-g",
    "infiniti-g35",
    "infiniti-g35-coupe",
    "infiniti-g37-convertible",
    "infiniti-g37-coupe",
    "infiniti-jx",
    "infiniti-m",
    "infiniti-q30",
    "infiniti-q45",
    "infiniti-q50",
    "infiniti-q60-convertible",
    "infiniti-q60-coupe",
    "infiniti-q70",
    "infiniti-qx30",
    "infiniti-qx50",
    "infiniti-qx55",
    "infiniti-qx56",
    "infiniti-qx60",
    "infiniti-qx70",
    "infiniti-qx80"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering infiniti models from %s", infiniti_INDEX)
    soup = fetch(session, infiniti_INDEX)
    if not soup:
        log.error("Failed to fetch infiniti index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/infiniti/(infiniti-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/infiniti/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/infiniti.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_infiniti import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["infiniti-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape infiniti vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/infiniti.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:13:29 [INFO] Discovering infiniti models from https://www.drivearabia.com/carprices/uae/infiniti/
15:13:29 [INFO] Discovered 22 models
15:13:29 [INFO] Year range: 2022â€“2026  |  Models: 22  |  Total pages: 110
15:13:32 [INFO] [1/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-ex/2022/
15:13:33 [INFO] [2/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-ex/2023/
15:13:34 [INFO] [3/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-ex/2024/
15:13:35 [INFO] [4/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-ex/2025/
15:13:37 [INFO] [5/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-ex/2026/
15:13:38 [INFO] [6/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-fx/2022/
15:13:39 [INFO] [7/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-fx/2023/
15:13:40 [INFO] [8/110] https://www.drivearabia.com/carprices/uae/infiniti/infiniti-fx/2024/
15:13:41 [INFO] [9/110] https://www.drivea

In [10]:
"""
DriveArabia UAE volkswagen Price Scraper
=====================================
Scrapes all volkswagen models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/volkswagen/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_volkswagen.py
    python scrape_drivearabia_volkswagen.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_volkswagen.py --models volkswagen-land-cruiser volkswagen-camry
    python scrape_drivearabia_volkswagen.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_volkswagen.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
volkswagen_INDEX = f"{BASE_URL}/carprices/uae/volkswagen/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"volkswagen|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|volkswagen)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": volkswagen_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "volkswagen-amarok",
    "volkswagen-arteon",
    "volkswagen-beetle",
    "volkswagen-beetle-cabriolet",
    "volkswagen-eos",
    "volkswagen-golf",
    "volkswagen-golf-gti",
    "volkswagen-golf-plus",
    "volkswagen-golf-r",
    "volkswagen-golf-r32",
    "volkswagen-jetta",
    "volkswagen-multivan",
    "volkswagen-passat",
    "volkswagen-passat-cc",
    "volkswagen-phaeton",
    "volkswagen-polo",
    "volkswagen-polo-sedan",
    "volkswagen-scirocco",
    "volkswagen-scirocco-r",
    "volkswagen-sharan",
    "volkswagen-t-roc",
    "volkswagen-teramont",
    "volkswagen-tiguan",
    "volkswagen-touareg",
    "volkswagen-touran"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering volkswagen models from %s", volkswagen_INDEX)
    soup = fetch(session, volkswagen_INDEX)
    if not soup:
        log.error("Failed to fetch volkswagen index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/volkswagen/(volkswagen-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/volkswagen/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/volkswagen.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_volkswagen import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["volkswagen-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape volkswagen vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/volkswagen.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:16:14 [INFO] Discovering volkswagen models from https://www.drivearabia.com/carprices/uae/volkswagen/
15:16:15 [INFO] Discovered 25 models
15:16:15 [INFO] Year range: 2022â€“2026  |  Models: 25  |  Total pages: 125
15:16:17 [INFO] [1/125] https://www.drivearabia.com/carprices/uae/volkswagen/volkswagen-amarok/2022/
15:16:18 [WARNING]   â†’ 0 trims parsed â€” try --debug-html to inspect page structure
15:16:19 [INFO] [2/125] https://www.drivearabia.com/carprices/uae/volkswagen/volkswagen-amarok/2023/
15:16:19 [INFO]   â†’ 4 trims
15:16:21 [INFO] [3/125] https://www.drivearabia.com/carprices/uae/volkswagen/volkswagen-amarok/2024/
15:16:21 [INFO]   â†’ 4 trims
15:16:23 [INFO] [4/125] https://www.drivearabia.com/carprices/uae/volkswagen/volkswagen-amarok/2025/
15:16:23 [INFO]   â†’ 4 trims
15:16:25 [INFO] [5/125] https://www.drivearabia.com/carprices/uae/volkswagen/volkswagen-amarok/2026/
15:16:25 [INFO]   â†’ 4 trims
15:16:26 [INFO] [6/125] https://www.drivearabia.com/carprices/uae/volk

In [11]:
"""
DriveArabia UAE suzuki Price Scraper
=====================================
Scrapes all suzuki models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/suzuki/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_suzuki.py
    python scrape_drivearabia_suzuki.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_suzuki.py --models suzuki-land-cruiser suzuki-camry
    python scrape_drivearabia_suzuki.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_suzuki.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
suzuki_INDEX = f"{BASE_URL}/carprices/uae/suzuki/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"suzuki|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|suzuki)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": suzuki_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
KNOWN_MODELS = [
    "suzuki-apv",
    "suzuki-across",
    "suzuki-alto",
    "suzuki-baleno",
    "suzuki-carry",
    "suzuki-celerio",
    "suzuki-ciaz",
    "suzuki-dzire",
    "suzuki-ertiga",
    "suzuki-fronx",
    "suzuki-grand-vitara",
    "suzuki-ignis",
    "suzuki-ignis-crossover",
    "suzuki-jimny",
    "suzuki-jimny-5-door",
    "suzuki-kizashi",
    "suzuki-liana",
    "suzuki-sx4",
    "suzuki-swift",
    "suzuki-swift-dzire",
    "suzuki-vitara",
    "suzuki-xl7"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering suzuki models from %s", suzuki_INDEX)
    soup = fetch(session, suzuki_INDEX)
    if not soup:
        log.error("Failed to fetch suzuki index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/suzuki/(suzuki-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/suzuki/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/suzuki.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_suzuki import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["suzuki-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape suzuki vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/suzuki.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:19:14 [INFO] Discovering suzuki models from https://www.drivearabia.com/carprices/uae/suzuki/
15:19:14 [INFO] Discovered 22 models
15:19:14 [INFO] Year range: 2022â€“2026  |  Models: 22  |  Total pages: 110
15:19:17 [INFO] [1/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-across/2022/
15:19:19 [INFO] [2/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-across/2023/
15:19:20 [INFO] [3/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-across/2024/
15:19:21 [INFO] [4/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-across/2025/
15:19:23 [INFO] [5/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-across/2026/
15:19:24 [INFO] [6/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-alto/2022/
15:19:26 [INFO] [7/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-alto/2023/
15:19:27 [INFO] [8/110] https://www.drivearabia.com/carprices/uae/suzuki/suzuki-alto/2024/
15:19:28 [INFO] [9/110] https://www.drivearabia.com/

In [12]:
"""
DriveArabia UAE abarth Price Scraper
=====================================
Scrapes all abarth models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/abarth/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_abarth.py
    python scrape_drivearabia_abarth.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_abarth.py --models abarth-land-cruiser abarth-camry
    python scrape_drivearabia_abarth.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_abarth.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
abarth_INDEX = f"{BASE_URL}/carprices/uae/abarth/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"abarth|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|abarth)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": abarth_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "abarth-124-spider",
    "abarth-595-competizione",
    "abarth-595-scorpionearo",
    "abarth-695"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering abarth models from %s", abarth_INDEX)
    soup = fetch(session, abarth_INDEX)
    if not soup:
        log.error("Failed to fetch abarth index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/abarth/(abarth-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/abarth/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/abarth.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_abarth import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["abarth-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape abarth vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/abarth.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:31:03 [INFO] Discovering abarth models from https://www.drivearabia.com/carprices/uae/abarth/
15:31:03 [INFO] Discovered 4 models
15:31:03 [INFO] Year range: 2022â€“2026  |  Models: 4  |  Total pages: 20
15:31:06 [INFO] [1/20] https://www.drivearabia.com/carprices/uae/abarth/abarth-124-spider/2022/
15:31:07 [INFO] [2/20] https://www.drivearabia.com/carprices/uae/abarth/abarth-124-spider/2023/
15:31:09 [INFO] [3/20] https://www.drivearabia.com/carprices/uae/abarth/abarth-124-spider/2024/
15:31:10 [INFO] [4/20] https://www.drivearabia.com/carprices/uae/abarth/abarth-124-spider/2025/
15:31:11 [INFO] [5/20] https://www.drivearabia.com/carprices/uae/abarth/abarth-124-spider/2026/
15:31:12 [INFO] [6/20] https://www.drivearabia.com/carprices/uae/abarth/abarth-595-competizione/2022/
15:31:13 [INFO]   â†’ 2 trims
15:31:14 [INFO] [7/20] https://www.drivearabia.com/carprices/uae/abarth/abarth-595-competizione/2023/
15:31:15 [INFO]   â†’ 2 trims
15:31:16 [INFO] [8/20] https://www.drivearabia.co

In [13]:
"""
DriveArabia UAE bmw Price Scraper
=====================================
Scrapes all bmw models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/bmw/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_bmw.py
    python scrape_drivearabia_bmw.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_bmw.py --models bmw-land-cruiser bmw-camry
    python scrape_drivearabia_bmw.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_bmw.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
bmw_INDEX = f"{BASE_URL}/carprices/uae/bmw/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"bmw|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|bmw)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": bmw_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "bmw-1-series",
    "bmw-2-series",
    "bmw-2-series-active-tourer",
    "bmw-2-series-convertible",
    "bmw-2-series-gran-coupe",
    "bmw-3-series",
    "bmw-3-series-convertible",
    "bmw-3-series-coupe",
    "bmw-4-series",
    "bmw-4-series-convertible",
    "bmw-4-series-gran-coupe",
    "bmw-5-series",
    "bmw-5-series-gt",
    "bmw-6-series-cabriolet",
    "bmw-6-series-coupe",
    "bmw-6-series-gran-coupe",
    "bmw-6-series-gran-turismo",
    "bmw-7-series",
    "bmw-8-series",
    "bmw-8-series-convertible",
    "bmw-8-series-gran-coupe",
    "bmw-m2",
    "bmw-m3",
    "bmw-m3-convertible",
    "bmw-m3-coupe",
    "bmw-m4",
    "bmw-m4-convertible",
    "bmw-m5",
    "bmw-m6-cabriolet",
    "bmw-m6-coupe",
    "bmw-m6-gran-coupe",
    "bmw-m8",
    "bmw-m8-convertible",
    "bmw-m8-gran-coupe",
    "bmw-x1",
    "bmw-x2",
    "bmw-x3",
    "bmw-x4",
    "bmw-x5",
    "bmw-x5-m",
    "bmw-x6",
    "bmw-x6-m",
    "bmw-x7",
    "bmw-xm",
    "bmw-z4",
    "bmw-i5",
    "bmw-i7",
    "bmw-i8",
    "bmw-ix",
    "bmw-ix1"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering bmw models from %s", bmw_INDEX)
    soup = fetch(session, bmw_INDEX)
    if not soup:
        log.error("Failed to fetch bmw index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/bmw/(bmw-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/bmw/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/bmw.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_bmw import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["bmw-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape bmw vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/bmw.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:37:28 [INFO] Discovering bmw models from https://www.drivearabia.com/carprices/uae/bmw/
15:37:29 [INFO] Discovered 50 models
15:37:29 [INFO] Year range: 2022â€“2026  |  Models: 50  |  Total pages: 250
15:37:32 [INFO] [1/250] https://www.drivearabia.com/carprices/uae/bmw/bmw-1-series/2022/
15:37:33 [INFO] [2/250] https://www.drivearabia.com/carprices/uae/bmw/bmw-1-series/2023/
15:37:34 [INFO] [3/250] https://www.drivearabia.com/carprices/uae/bmw/bmw-1-series/2024/
15:37:35 [INFO]   â†’ 1 trims
15:37:36 [INFO] [4/250] https://www.drivearabia.com/carprices/uae/bmw/bmw-1-series/2025/
15:37:36 [INFO]   â†’ 1 trims
15:37:38 [INFO] [5/250] https://www.drivearabia.com/carprices/uae/bmw/bmw-1-series/2026/
15:37:38 [INFO]   â†’ 1 trims
15:37:40 [INFO] [6/250] https://www.drivearabia.com/carprices/uae/bmw/bmw-2-series/2022/
15:37:40 [WARNING]   â†’ 0 trims parsed â€” try --debug-html to inspect page structure
15:37:42 [INFO] [7/250] https://www.drivearabia.com/carprices/uae/bmw/bmw-2-series/20

In [17]:
"""
DriveArabia UAE mitsubishi Price Scraper
=====================================
Scrapes all mitsubishi models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/mitsubishi/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_mitsubishi.py
    python scrape_drivearabia_mitsubishi.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_mitsubishi.py --models mitsubishi-land-cruiser mitsubishi-camry
    python scrape_drivearabia_mitsubishi.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_mitsubishi.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
mitsubishi_INDEX = f"{BASE_URL}/carprices/uae/mitsubishi/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"gmc|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|mitsubishi)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": mitsubishi_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "mitsubishi-asx",
    "mitsubishi-attrage",
    "mitsubishi-destinator",
    "mitsubishi-eclipse",
    "mitsubishi-eclipse-cross",
    "mitsubishi-galant",
    "mitsubishi-grandis",
    "mitsubishi-l200",
    "mitsubishi-lancer",
    "mitsubishi-lancer-ex",
    "mitsubishi-lancer-evolution",
    "mitsubishi-lancer-fortis",
    "mitsubishi-magna",
    "mitsubishi-mirage",
    "mitsubishi-montero-sport",
    "mitsubishi-nativa",
    "mitsubishi-outlander",
    "mitsubishi-outlander-phev",
    "mitsubishi-pajero",
    "mitsubishi-pajero-swb",
    "mitsubishi-pajero-sport",
    "mitsubishi-xpander",
    "mitsubishi-xpander-cross"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering mitsubishi models from %s", mitsubishi_INDEX)
    soup = fetch(session, mitsubishi_INDEX)
    if not soup:
        log.error("Failed to fetch mitsubishi index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/mitsubishi/(mitsubishi-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/mitsubishi/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/drivearabia_mitsubishi_prices.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_mitsubishi import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["mitsubishi-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape mitsubishi vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/drivearabia_mitsubishi_prices.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:45:26 [INFO] Discovering mitsubishi models from https://www.drivearabia.com/carprices/uae/mitsubishi/
15:45:28 [INFO] Discovered 23 models
15:45:28 [INFO] Year range: 2022â€“2026  |  Models: 23  |  Total pages: 115
15:45:30 [INFO] [1/115] https://www.drivearabia.com/carprices/uae/mitsubishi/mitsubishi-asx/2022/
15:45:30 [INFO]   â†’ 3 trims
15:45:32 [INFO] [2/115] https://www.drivearabia.com/carprices/uae/mitsubishi/mitsubishi-asx/2023/
15:45:32 [INFO]   â†’ 3 trims
15:45:34 [INFO] [3/115] https://www.drivearabia.com/carprices/uae/mitsubishi/mitsubishi-asx/2024/
15:45:34 [INFO]   â†’ 3 trims
15:45:36 [INFO] [4/115] https://www.drivearabia.com/carprices/uae/mitsubishi/mitsubishi-asx/2025/
15:45:36 [INFO]   â†’ 2 trims
15:45:37 [INFO] [5/115] https://www.drivearabia.com/carprices/uae/mitsubishi/mitsubishi-asx/2026/
15:45:38 [INFO]   â†’ 2 trims
15:45:39 [INFO] [6/115] https://www.drivearabia.com/carprices/uae/mitsubishi/mitsubishi-attrage/2022/
15:45:40 [INFO]   â†’ 3 trims
15:45:41 [

In [18]:
"""
DriveArabia UAE lexus Price Scraper
=====================================
Scrapes all lexus models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/lexus/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_lexus.py
    python scrape_drivearabia_lexus.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_lexus.py --models lexus-land-cruiser lexus-camry
    python scrape_drivearabia_lexus.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_lexus.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
lexus_INDEX = f"{BASE_URL}/carprices/uae/lexus/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"lexus|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|lexus)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": lexus_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "lexus-ct",
    "lexus-es",
    "lexus-gs",
    "lexus-gx",
    "lexus-is",
    "lexus-is-c",
    "lexus-is-f",
    "lexus-lc",
    "lexus-lfa",
    "lexus-ls",
    "lexus-lx",
    "lexus-nx",
    "lexus-rc",
    "lexus-rc-f",
    "lexus-rx",
    "lexus-rx-hybrid",
    "lexus-sc",
    "lexus-ux",
    "lexus-ux-hybrid"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering lexus models from %s", lexus_INDEX)
    soup = fetch(session, lexus_INDEX)
    if not soup:
        log.error("Failed to fetch lexus index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/lexus/(lexus-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/lexus/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/lexus.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_lexus import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["lexus-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape lexus vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/lexus.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:48:17 [INFO] Discovering lexus models from https://www.drivearabia.com/carprices/uae/lexus/
15:48:19 [INFO] Discovered 19 models
15:48:19 [INFO] Year range: 2022â€“2026  |  Models: 19  |  Total pages: 95
15:48:20 [INFO] [1/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2022/
15:48:21 [INFO] [2/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2023/
15:48:23 [INFO] [3/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2024/
15:48:24 [INFO] [4/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2025/
15:48:25 [INFO] [5/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2026/
15:48:26 [INFO] [6/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-es/2022/
15:48:26 [INFO]   â†’ 6 trims
15:48:28 [INFO] [7/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-es/2023/
15:48:28 [INFO]   â†’ 4 trims
15:48:30 [INFO] [8/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-es/2024/
15:48:30 [INFO]   â†’ 4 trims
15:48:32 [INFO]

In [19]:
"""
DriveArabia UAE tesla Price Scraper
=====================================
Scrapes all tesla models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/tesla/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_tesla.py
    python scrape_drivearabia_tesla.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_tesla.py --models tesla-land-cruiser tesla-camry
    python scrape_drivearabia_tesla.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_tesla.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
tesla_INDEX = f"{BASE_URL}/carprices/uae/tesla/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"tesla|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"tesla|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|tesla)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": tesla_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "tesla-cybertruck",
    "tesla-model-3",
    "tesla-model-s",
    "tesla-model-x",
    "tesla-model-y"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering tesla models from %s", tesla_INDEX)
    soup = fetch(session, tesla_INDEX)
    if not soup:
        log.error("Failed to fetch tesla index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/tesla/(tesla-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/tesla/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/tesla.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_tesla import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["tesla-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape tesla vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/tesla.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:50:47 [INFO] Discovering tesla models from https://www.drivearabia.com/carprices/uae/tesla/
15:50:48 [INFO] Discovered 5 models
15:50:48 [INFO] Year range: 2022â€“2026  |  Models: 5  |  Total pages: 25
15:50:50 [INFO] [1/25] https://www.drivearabia.com/carprices/uae/tesla/tesla-cybertruck/2022/
15:50:51 [INFO] [2/25] https://www.drivearabia.com/carprices/uae/tesla/tesla-cybertruck/2023/
15:50:52 [INFO] [3/25] https://www.drivearabia.com/carprices/uae/tesla/tesla-cybertruck/2024/
15:50:54 [INFO] [4/25] https://www.drivearabia.com/carprices/uae/tesla/tesla-cybertruck/2025/
15:50:54 [INFO]   â†’ 2 trims
15:50:56 [INFO] [5/25] https://www.drivearabia.com/carprices/uae/tesla/tesla-cybertruck/2026/
15:50:56 [INFO]   â†’ 2 trims
15:50:58 [INFO] [6/25] https://www.drivearabia.com/carprices/uae/tesla/tesla-model-3/2022/
15:50:58 [INFO]   â†’ 3 trims
15:51:00 [INFO] [7/25] https://www.drivearabia.com/carprices/uae/tesla/tesla-model-3/2023/
15:51:00 [INFO]   â†’ 3 trims
15:51:02 [INFO] [8/25] 

In [20]:
"""
DriveArabia UAE lexus Price Scraper
=====================================
Scrapes all lexus models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/lexus/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_lexus.py
    python scrape_drivearabia_lexus.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_lexus.py --models lexus-land-cruiser lexus-camry
    python scrape_drivearabia_lexus.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_lexus.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
lexus_INDEX = f"{BASE_URL}/carprices/uae/lexus/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lexus|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"lexus|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|lexus)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": lexus_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "lexus-ct",
    "lexus-es",
    "lexus-gs",
    "lexus-gx",
    "lexus-is",
    "lexus-is-c",
    "lexus-is-f",
    "lexus-lc",
    "lexus-lfa",
    "lexus-ls",
    "lexus-lx",
    "lexus-nx",
    "lexus-rc",
    "lexus-rc-f",
    "lexus-rx",
    "lexus-rx-hybrid",
    "lexus-sc",
    "lexus-ux",
    "lexus-ux-hybrid"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering lexus models from %s", lexus_INDEX)
    soup = fetch(session, lexus_INDEX)
    if not soup:
        log.error("Failed to fetch lexus index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/lexus/(lexus-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/lexus/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/lexus.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_lexus import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["lexus-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape lexus vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/lexus.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:51:40 [INFO] Discovering lexus models from https://www.drivearabia.com/carprices/uae/lexus/
15:51:41 [INFO] Discovered 19 models
15:51:41 [INFO] Year range: 2022â€“2026  |  Models: 19  |  Total pages: 95
15:51:42 [INFO] [1/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2022/
15:51:43 [INFO] [2/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2023/
15:51:44 [INFO] [3/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2024/
15:51:45 [INFO] [4/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2025/
15:51:46 [INFO] [5/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-ct/2026/
15:51:47 [INFO] [6/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-es/2022/
15:51:48 [INFO]   â†’ 6 trims
15:51:49 [INFO] [7/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-es/2023/
15:51:50 [INFO]   â†’ 4 trims
15:51:51 [INFO] [8/95] https://www.drivearabia.com/carprices/uae/lexus/lexus-es/2024/
15:51:51 [INFO]   â†’ 4 trims
15:51:53 [INFO]

In [21]:
"""
DriveArabia UAE jeep Price Scraper
=====================================
Scrapes all jeep models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/jeep/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_jeep.py
    python scrape_drivearabia_jeep.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_jeep.py --models jeep-land-cruiser jeep-camry
    python scrape_drivearabia_jeep.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_jeep.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
jeep_INDEX = f"{BASE_URL}/carprices/uae/jeep/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"jeep|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"jeep|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|jeep)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": jeep_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "jeep-cherokee",
    "jeep-commander",
    "jeep-compass",
    "jeep-gladiator",
    "jeep-grand-cherokee",
    "jeep-grand-cherokee-l",
    "jeep-grand-cherokee-srt",
    "jeep-grand-cherokee-trackhawk",
    "jeep-grand-wagoneer",
    "jeep-patriot",
    "jeep-renegade",
    "jeep-wrangler",
    "jeep-wrangler-rubicon-392",
    "jeep-wrangler-unlimited"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering jeep models from %s", jeep_INDEX)
    soup = fetch(session, jeep_INDEX)
    if not soup:
        log.error("Failed to fetch jeep index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/jeep/(jeep-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/jeep/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/jeep.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_jeep import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["jeep-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape jeep vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/jeep.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



15:54:25 [INFO] Discovering jeep models from https://www.drivearabia.com/carprices/uae/jeep/
15:54:36 [WARNING] HTTP 500 â€” https://www.drivearabia.com/carprices/uae/jeep/ (attempt 1)
15:54:37 [INFO] Discovered 14 models
15:54:37 [INFO] Year range: 2022â€“2026  |  Models: 14  |  Total pages: 70
15:54:39 [INFO] [1/70] https://www.drivearabia.com/carprices/uae/jeep/jeep-cherokee/2022/
15:54:39 [INFO]   â†’ 4 trims
15:54:40 [INFO] [2/70] https://www.drivearabia.com/carprices/uae/jeep/jeep-cherokee/2023/
15:54:42 [INFO] [3/70] https://www.drivearabia.com/carprices/uae/jeep/jeep-cherokee/2024/
15:54:43 [INFO] [4/70] https://www.drivearabia.com/carprices/uae/jeep/jeep-cherokee/2025/
15:54:44 [INFO] [5/70] https://www.drivearabia.com/carprices/uae/jeep/jeep-cherokee/2026/
15:54:45 [INFO] [6/70] https://www.drivearabia.com/carprices/uae/jeep/jeep-commander/2022/
15:54:47 [INFO] [7/70] https://www.drivearabia.com/carprices/uae/jeep/jeep-commander/2023/
15:54:48 [INFO] [8/70] https://www.drivea

In [22]:
"""
DriveArabia UAE dodge Price Scraper
=====================================
Scrapes all dodge models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/dodge/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_dodge.py
    python scrape_drivearabia_dodge.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_dodge.py --models dodge-land-cruiser dodge-camry
    python scrape_drivearabia_dodge.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_dodge.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
dodge_INDEX = f"{BASE_URL}/carprices/uae/dodge/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"dodge|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"dodge|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|dodge)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": dodge_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "dodge-avenger",
    "dodge-caliber",
    "dodge-challenger",
    "dodge-challenger-srt",
    "dodge-charger",
    "dodge-charger-srt",
    "dodge-dakota",
    "dodge-dart",
    "dodge-durango",
    "dodge-durango-srt",
    "dodge-neon",
    "dodge-nitro",
    "dodge-ram",
    "dodge-viper"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering dodge models from %s", dodge_INDEX)
    soup = fetch(session, dodge_INDEX)
    if not soup:
        log.error("Failed to fetch dodge index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/dodge/(dodge-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/dodge/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/dodge.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_dodge import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["dodge-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape dodge vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/dodge.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:02:10 [INFO] Discovering dodge models from https://www.drivearabia.com/carprices/uae/dodge/
16:02:11 [INFO] Discovered 14 models
16:02:11 [INFO] Year range: 2022â€“2026  |  Models: 14  |  Total pages: 70
16:02:12 [INFO] [1/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-avenger/2022/
16:02:13 [INFO] [2/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-avenger/2023/
16:02:15 [INFO] [3/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-avenger/2024/
16:02:16 [INFO] [4/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-avenger/2025/
16:02:17 [INFO] [5/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-avenger/2026/
16:02:18 [INFO] [6/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-caliber/2022/
16:02:19 [INFO] [7/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-caliber/2023/
16:02:20 [INFO] [8/70] https://www.drivearabia.com/carprices/uae/dodge/dodge-caliber/2024/
16:02:22 [INFO] [9/70] https://www.drivearabia.com/carprices/uae/

In [23]:
"""
DriveArabia UAE changan Price Scraper
=====================================
Scrapes all changan models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/changan/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_changan.py
    python scrape_drivearabia_changan.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_changan.py --models changan-land-cruiser changan-camry
    python scrape_drivearabia_changan.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_changan.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
changan_INDEX = f"{BASE_URL}/carprices/uae/changan/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"changan|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"changan|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|changan)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": changan_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "changan-alsvin",
    "changan-cs35",
    "changan-cs35-plus",
    "changan-cs75",
    "changan-cs75-plus",
    "changan-cs85",
    "changan-cs95",
    "changan-eado",
    "changan-uni-k",
    "changan-uni-t",
    "changan-uni-v"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering changan models from %s", changan_INDEX)
    soup = fetch(session, changan_INDEX)
    if not soup:
        log.error("Failed to fetch changan index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/changan/(changan-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/changan/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/changan.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_changan import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["changan-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape changan vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/changan.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:03:48 [INFO] Discovering changan models from https://www.drivearabia.com/carprices/uae/changan/
16:03:48 [INFO] Discovered 11 models
16:03:48 [INFO] Year range: 2022â€“2026  |  Models: 11  |  Total pages: 55
16:03:50 [INFO] [1/55] https://www.drivearabia.com/carprices/uae/changan/changan-alsvin/2022/
16:03:50 [INFO]   â†’ 2 trims
16:03:52 [INFO] [2/55] https://www.drivearabia.com/carprices/uae/changan/changan-alsvin/2023/
16:03:52 [INFO]   â†’ 2 trims
16:03:54 [INFO] [3/55] https://www.drivearabia.com/carprices/uae/changan/changan-alsvin/2024/
16:03:54 [INFO]   â†’ 1 trims
16:03:56 [INFO] [4/55] https://www.drivearabia.com/carprices/uae/changan/changan-alsvin/2025/
16:03:56 [INFO]   â†’ 1 trims
16:03:58 [INFO] [5/55] https://www.drivearabia.com/carprices/uae/changan/changan-alsvin/2026/
16:03:58 [INFO]   â†’ 1 trims
16:04:00 [INFO] [6/55] https://www.drivearabia.com/carprices/uae/changan/changan-cs35/2022/
16:04:01 [INFO] [7/55] https://www.drivearabia.com/carprices/uae/changan/chan

In [24]:
"""
DriveArabia UAE renault Price Scraper
=====================================
Scrapes all renault models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/renault/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_renault.py
    python scrape_drivearabia_renault.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_renault.py --models renault-land-cruiser renault-camry
    python scrape_drivearabia_renault.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_renault.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
renault_INDEX = f"{BASE_URL}/carprices/uae/renault/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"renault|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"renault|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|renault)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": renault_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "renault-arkana",
    "renault-captur",
    "renault-clio",
    "renault-clio-rs",
    "renault-dokker-van",
    "renault-duster",
    "renault-espace",
    "renault-express-van",
    "renault-fluence",
    "renault-koleos",
    "renault-laguna",
    "renault-laguna-coupe",
    "renault-logan",
    "renault-logan-van",
    "renault-master",
    "renault-megane",
    "renault-megane-cc",
    "renault-megane-gt",
    "renault-megane-rs",
    "renault-safrane",
    "renault-sandero",
    "renault-scenic",
    "renault-symbol",
    "renault-talisman",
    "renault-trafic",
    "renault-twizy",
    "renault-zoe"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering renault models from %s", renault_INDEX)
    soup = fetch(session, renault_INDEX)
    if not soup:
        log.error("Failed to fetch renault index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/renault/(renault-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/renault/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/renault.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_renault import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["renault-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape renault vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/renault.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:07:24 [INFO] Discovering renault models from https://www.drivearabia.com/carprices/uae/renault/
16:07:24 [INFO] Discovered 27 models
16:07:24 [INFO] Year range: 2022â€“2026  |  Models: 27  |  Total pages: 135
16:07:26 [INFO] [1/135] https://www.drivearabia.com/carprices/uae/renault/renault-arkana/2022/
16:07:27 [INFO] [2/135] https://www.drivearabia.com/carprices/uae/renault/renault-arkana/2023/
16:07:28 [INFO] [3/135] https://www.drivearabia.com/carprices/uae/renault/renault-arkana/2024/
16:07:29 [INFO] [4/135] https://www.drivearabia.com/carprices/uae/renault/renault-arkana/2025/
16:07:30 [INFO]   â†’ 1 trims
16:07:31 [INFO] [5/135] https://www.drivearabia.com/carprices/uae/renault/renault-arkana/2026/
16:07:32 [INFO]   â†’ 1 trims
16:07:33 [INFO] [6/135] https://www.drivearabia.com/carprices/uae/renault/renault-captur/2022/
16:07:35 [INFO] [7/135] https://www.drivearabia.com/carprices/uae/renault/renault-captur/2023/
16:07:36 [INFO] [8/135] https://www.drivearabia.com/carprices/u

In [25]:
"""
DriveArabia UAE porsche Price Scraper
=====================================
Scrapes all porsche models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/porsche/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_porsche.py
    python scrape_drivearabia_porsche.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_porsche.py --models porsche-land-cruiser porsche-camry
    python scrape_drivearabia_porsche.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_porsche.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
porsche_INDEX = f"{BASE_URL}/carprices/uae/porsche/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"porsche|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"porsche|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|porsche)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": porsche_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "porsche-911",
    "porsche-911-cabriolet",
    "porsche-911-gt2",
    "porsche-911-gt3",
    "porsche-911-targa",
    "porsche-911-turbo",
    "porsche-911-turbo-cabriolet",
    "porsche-918-spyder",
    "porsche-boxster",
    "porsche-cayenne",
    "porsche-cayenne-coupe",
    "porsche-cayenne-electric",
    "porsche-cayman",
    "porsche-macan",
    "porsche-panamera",
    "porsche-panamera-sport-turismo",
    "porsche-taycan",
    "porsche-taycan-turbo"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering porsche models from %s", porsche_INDEX)
    soup = fetch(session, porsche_INDEX)
    if not soup:
        log.error("Failed to fetch porsche index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/porsche/(porsche-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/porsche/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/porsche.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_porsche import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["porsche-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape porsche vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/porsche.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:11:59 [INFO] Discovering porsche models from https://www.drivearabia.com/carprices/uae/porsche/
16:12:00 [INFO] Discovered 18 models
16:12:00 [INFO] Year range: 2022â€“2026  |  Models: 18  |  Total pages: 90
16:12:01 [INFO] [1/90] https://www.drivearabia.com/carprices/uae/porsche/porsche-911/2022/
16:12:02 [INFO]   â†’ 6 trims
16:12:03 [INFO] [2/90] https://www.drivearabia.com/carprices/uae/porsche/porsche-911/2023/
16:12:04 [INFO]   â†’ 6 trims
16:12:05 [INFO] [3/90] https://www.drivearabia.com/carprices/uae/porsche/porsche-911/2024/
16:12:06 [INFO]   â†’ 6 trims
16:12:07 [INFO] [4/90] https://www.drivearabia.com/carprices/uae/porsche/porsche-911/2025/
16:12:08 [INFO]   â†’ 4 trims
16:12:09 [INFO] [5/90] https://www.drivearabia.com/carprices/uae/porsche/porsche-911/2026/
16:12:09 [INFO]   â†’ 4 trims
16:12:11 [INFO] [6/90] https://www.drivearabia.com/carprices/uae/porsche/porsche-911-cabriolet/2022/
16:12:11 [INFO]   â†’ 4 trims
16:12:13 [INFO] [7/90] https://www.drivearabia.com/ca

In [26]:
"""
DriveArabia UAE rox Price Scraper
=====================================
Scrapes all rox models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/rox/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_rox.py
    python scrape_drivearabia_rox.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_rox.py --models rox-land-cruiser rox-camry
    python scrape_drivearabia_rox.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_rox.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
rox_INDEX = f"{BASE_URL}/carprices/uae/rox/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"rox|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"rox|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|rox)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": rox_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "rox-01",
    "rox-adamas"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering rox models from %s", rox_INDEX)
    soup = fetch(session, rox_INDEX)
    if not soup:
        log.error("Failed to fetch rox index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/rox/(rox-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/rox/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/rox.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_rox import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["rox-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape rox vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/rox.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:15:05 [INFO] Discovering rox models from https://www.drivearabia.com/carprices/uae/rox/
16:15:06 [INFO] Discovered 2 models
16:15:06 [INFO] Year range: 2022â€“2026  |  Models: 2  |  Total pages: 10
16:15:07 [INFO] [1/10] https://www.drivearabia.com/carprices/uae/rox/rox-01/2022/
16:15:08 [INFO] [2/10] https://www.drivearabia.com/carprices/uae/rox/rox-01/2023/
16:15:09 [INFO] [3/10] https://www.drivearabia.com/carprices/uae/rox/rox-01/2024/
16:15:10 [INFO] [4/10] https://www.drivearabia.com/carprices/uae/rox/rox-01/2025/
16:15:10 [INFO]   â†’ 1 trims
16:15:12 [INFO] [5/10] https://www.drivearabia.com/carprices/uae/rox/rox-01/2026/
16:15:12 [INFO]   â†’ 1 trims
16:15:14 [INFO] [6/10] https://www.drivearabia.com/carprices/uae/rox/rox-adamas/2022/
16:15:15 [INFO] [7/10] https://www.drivearabia.com/carprices/uae/rox/rox-adamas/2023/
16:15:16 [INFO] [8/10] https://www.drivearabia.com/carprices/uae/rox/rox-adamas/2024/
16:15:18 [INFO] [9/10] https://www.drivearabia.com/carprices/uae/rox/ro

In [27]:
"""
DriveArabia UAE lincoln Price Scraper
=====================================
Scrapes all lincoln models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/lincoln/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_lincoln.py
    python scrape_drivearabia_lincoln.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_lincoln.py --models lincoln-land-cruiser lincoln-camry
    python scrape_drivearabia_lincoln.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_lincoln.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
lincoln_INDEX = f"{BASE_URL}/carprices/uae/lincoln/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"lincoln|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"lincoln|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|lincoln)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": lincoln_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "lincoln-aviator",
    "lincoln-continental",
    "lincoln-corsair",
    "lincoln-mkc",
    "lincoln-mks",
    "lincoln-mkt",
    "lincoln-mkx",
    "lincoln-mkz",
    "lincoln-nautilus",
    "lincoln-navigator",
    "lincoln-navigator-l",
    "lincoln-town-car"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering lincoln models from %s", lincoln_INDEX)
    soup = fetch(session, lincoln_INDEX)
    if not soup:
        log.error("Failed to fetch lincoln index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/lincoln/(lincoln-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/lincoln/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/lincoln.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_lincoln import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["lincoln-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape lincoln vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/lincoln.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:17:33 [INFO] Discovering lincoln models from https://www.drivearabia.com/carprices/uae/lincoln/
16:17:34 [INFO] Discovered 12 models
16:17:34 [INFO] Year range: 2022â€“2026  |  Models: 12  |  Total pages: 60
16:17:35 [INFO] [1/60] https://www.drivearabia.com/carprices/uae/lincoln/lincoln-aviator/2022/
16:17:36 [INFO]   â†’ 4 trims
16:17:37 [INFO] [2/60] https://www.drivearabia.com/carprices/uae/lincoln/lincoln-aviator/2023/
16:17:38 [INFO]   â†’ 4 trims
16:17:39 [INFO] [3/60] https://www.drivearabia.com/carprices/uae/lincoln/lincoln-aviator/2024/
16:17:40 [INFO]   â†’ 4 trims
16:17:41 [INFO] [4/60] https://www.drivearabia.com/carprices/uae/lincoln/lincoln-aviator/2025/
16:17:42 [INFO]   â†’ 2 trims
16:17:43 [INFO] [5/60] https://www.drivearabia.com/carprices/uae/lincoln/lincoln-aviator/2026/
16:17:44 [INFO]   â†’ 2 trims
16:17:45 [INFO] [6/60] https://www.drivearabia.com/carprices/uae/lincoln/lincoln-continental/2022/
16:17:47 [INFO] [7/60] https://www.drivearabia.com/carprices/uae/

In [28]:
"""
DriveArabia UAE peugeot Price Scraper
=====================================
Scrapes all peugeot models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/peugeot/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_peugeot.py
    python scrape_drivearabia_peugeot.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_peugeot.py --models peugeot-land-cruiser peugeot-camry
    python scrape_drivearabia_peugeot.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_peugeot.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
peugeot_INDEX = f"{BASE_URL}/carprices/uae/peugeot/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"peugeot|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"peugeot|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|peugeot)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": peugeot_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "peugeot-2008",
    "peugeot-206",
    "peugeot-207",
    "peugeot-208",
    "peugeot-3008",
    "peugeot-3008-hybrid-4",
    "peugeot-301",
    "peugeot-307",
    "peugeot-308",
    "peugeot-308-cc",
    "peugeot-407",
    "peugeot-407-coupe",
    "peugeot-408",
    "peugeot-5008",
    "peugeot-508",
    "peugeot-607",
    "peugeot-boxer",
    "peugeot-expert",
    "peugeot-landtrek",
    "peugeot-partner",
    "peugeot-rcz",
    "peugeot-traveller",
    "peugeot-e-2008",
    "peugeot-e-208"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering peugeot models from %s", peugeot_INDEX)
    soup = fetch(session, peugeot_INDEX)
    if not soup:
        log.error("Failed to fetch peugeot index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/peugeot/(peugeot-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/peugeot/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/peugeot.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_peugeot import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["peugeot-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape peugeot vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/peugeot.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:38:41 [INFO] Discovering peugeot models from https://www.drivearabia.com/carprices/uae/peugeot/
16:38:41 [INFO] Discovered 24 models
16:38:41 [INFO] Year range: 2022â€“2026  |  Models: 24  |  Total pages: 120
16:38:44 [INFO] [1/120] https://www.drivearabia.com/carprices/uae/peugeot/peugeot-2008/2022/
16:38:44 [INFO]   â†’ 3 trims
16:38:46 [INFO] [2/120] https://www.drivearabia.com/carprices/uae/peugeot/peugeot-2008/2023/
16:38:47 [INFO]   â†’ 3 trims
16:38:48 [INFO] [3/120] https://www.drivearabia.com/carprices/uae/peugeot/peugeot-2008/2024/
16:38:49 [INFO]   â†’ 3 trims
16:38:50 [INFO] [4/120] https://www.drivearabia.com/carprices/uae/peugeot/peugeot-2008/2025/
16:38:51 [INFO]   â†’ 3 trims
16:38:52 [INFO] [5/120] https://www.drivearabia.com/carprices/uae/peugeot/peugeot-2008/2026/
16:38:53 [INFO]   â†’ 3 trims
16:38:54 [INFO] [6/120] https://www.drivearabia.com/carprices/uae/peugeot/peugeot-206/2022/
16:38:55 [INFO] [7/120] https://www.drivearabia.com/carprices/uae/peugeot/peugeot

In [29]:
"""
DriveArabia UAE ram Price Scraper
=====================================
Scrapes all ram models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/ram/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_ram.py
    python scrape_drivearabia_ram.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_ram.py --models ram-land-cruiser ram-camry
    python scrape_drivearabia_ram.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_ram.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
ram_INDEX = f"{BASE_URL}/carprices/uae/ram/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"ram|mitsubishi|jeep|dodge|ram|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"ram|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|ram)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": ram_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "ram-1200",
    "ram-1500",
    "ram-1500-trx",
    "ram-2500"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering ram models from %s", ram_INDEX)
    soup = fetch(session, ram_INDEX)
    if not soup:
        log.error("Failed to fetch ram index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/ram/(ram-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/ram/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/ram.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_ram import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["ram-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape ram vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/ram.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:42:01 [INFO] Discovering ram models from https://www.drivearabia.com/carprices/uae/ram/
16:42:03 [INFO] Discovered 4 models
16:42:03 [INFO] Year range: 2022â€“2026  |  Models: 4  |  Total pages: 20
16:42:04 [INFO] [1/20] https://www.drivearabia.com/carprices/uae/ram/ram-1200/2022/
16:42:06 [INFO] [2/20] https://www.drivearabia.com/carprices/uae/ram/ram-1200/2023/
16:42:07 [INFO] [3/20] https://www.drivearabia.com/carprices/uae/ram/ram-1200/2024/
16:42:08 [INFO] [4/20] https://www.drivearabia.com/carprices/uae/ram/ram-1200/2025/
16:42:09 [INFO] [5/20] https://www.drivearabia.com/carprices/uae/ram/ram-1200/2026/
16:42:10 [INFO] [6/20] https://www.drivearabia.com/carprices/uae/ram/ram-1500/2022/
16:42:11 [INFO]   â†’ 4 trims
16:42:12 [INFO] [7/20] https://www.drivearabia.com/carprices/uae/ram/ram-1500/2023/
16:42:13 [INFO]   â†’ 4 trims
16:42:14 [INFO] [8/20] https://www.drivearabia.com/carprices/uae/ram/ram-1500/2024/
16:42:14 [INFO]   â†’ 3 trims
16:42:16 [INFO] [9/20] https://www.dr

In [30]:
"""
DriveArabia UAE geely Price Scraper
=====================================
Scrapes all geely models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/geely/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_geely.py
    python scrape_drivearabia_geely.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_geely.py --models geely-land-cruiser geely-camry
    python scrape_drivearabia_geely.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_geely.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
geely_INDEX = f"{BASE_URL}/carprices/uae/geely/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"geely|mitsubishi|jeep|dodge|geely|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"geely|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|geely)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": geely_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "geely-ck",
    "geely-cityray",
    "geely-coolray",
    "geely-ex5",
    "geely-emgrand",
    "geely-emgrand-7",
    "geely-emgrand-8",
    "geely-emgrand-gs",
    "geely-emgrand-gt",
    "geely-emgrand-x7",
    "geely-emgrand-x7-sport",
    "geely-gc2",
    "geely-gc6",
    "geely-gc7",
    "geely-gx2",
    "geely-gx3",
    "geely-gx3-pro",
    "geely-geometry-c",
    "geely-monjaro",
    "geely-okavango",
    "geely-preface",
    "geely-starray",
    "geely-tugella"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering geely models from %s", geely_INDEX)
    soup = fetch(session, geely_INDEX)
    if not soup:
        log.error("Failed to fetch geely index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/geely/(geely-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/geely/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/geely.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_geely import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["geely-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape geely vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/geely.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:42:41 [INFO] Discovering geely models from https://www.drivearabia.com/carprices/uae/geely/
16:42:43 [INFO] Discovered 23 models
16:42:43 [INFO] Year range: 2022â€“2026  |  Models: 23  |  Total pages: 115
16:42:45 [INFO] [1/115] https://www.drivearabia.com/carprices/uae/geely/geely-cityray/2022/
16:42:46 [INFO] [2/115] https://www.drivearabia.com/carprices/uae/geely/geely-cityray/2023/
16:42:47 [INFO] [3/115] https://www.drivearabia.com/carprices/uae/geely/geely-cityray/2024/
16:42:48 [INFO] [4/115] https://www.drivearabia.com/carprices/uae/geely/geely-cityray/2025/
16:42:49 [INFO] [5/115] https://www.drivearabia.com/carprices/uae/geely/geely-cityray/2026/
16:42:49 [INFO]   â†’ 3 trims
16:42:51 [INFO] [6/115] https://www.drivearabia.com/carprices/uae/geely/geely-ck/2022/
16:42:52 [INFO] [7/115] https://www.drivearabia.com/carprices/uae/geely/geely-ck/2023/
16:42:53 [INFO] [8/115] https://www.drivearabia.com/carprices/uae/geely/geely-ck/2024/
16:42:54 [INFO] [9/115] https://www.drive

In [31]:
"""
DriveArabia UAE cadillac Price Scraper
=====================================
Scrapes all cadillac models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/cadillac/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_cadillac.py
    python scrape_drivearabia_cadillac.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_cadillac.py --models cadillac-land-cruiser cadillac-camry
    python scrape_drivearabia_cadillac.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_cadillac.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
cadillac_INDEX = f"{BASE_URL}/carprices/uae/cadillac/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"cadillac|mitsubishi|jeep|dodge|cadillac|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"cadillac|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|cadillac)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": cadillac_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "cadillac-ats",
    "cadillac-ats-coupe",
    "cadillac-ats-v",
    "cadillac-ats-v-coupe",
    "cadillac-bls",
    "cadillac-ct4",
    "cadillac-ct4-v",
    "cadillac-ct5",
    "cadillac-ct5-v",
    "cadillac-ct6",
    "cadillac-cts",
    "cadillac-cts-coupe",
    "cadillac-cts-v",
    "cadillac-cts-v-coupe",
    "cadillac-dts",
    "cadillac-escalade",
    "cadillac-escalade-iq",
    "cadillac-escalade-v",
    "cadillac-lyriq",
    "cadillac-optiq",
    "cadillac-sls",
    "cadillac-srx",
    "cadillac-sts",
    "cadillac-xlr",
    "cadillac-xt4",
    "cadillac-xt5",
    "cadillac-xt6",
    "cadillac-xts"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering cadillac models from %s", cadillac_INDEX)
    soup = fetch(session, cadillac_INDEX)
    if not soup:
        log.error("Failed to fetch cadillac index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/cadillac/(cadillac-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/cadillac/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/cadillac.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_cadillac import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["cadillac-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape cadillac vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/cadillac.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:45:22 [INFO] Discovering cadillac models from https://www.drivearabia.com/carprices/uae/cadillac/
16:45:24 [INFO] Discovered 28 models
16:45:24 [INFO] Year range: 2022â€“2026  |  Models: 28  |  Total pages: 140
16:45:25 [INFO] [1/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats/2022/
16:45:26 [INFO] [2/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats/2023/
16:45:27 [INFO] [3/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats/2024/
16:45:28 [INFO] [4/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats/2025/
16:45:30 [INFO] [5/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats/2026/
16:45:31 [INFO] [6/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats-coupe/2022/
16:45:32 [INFO] [7/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats-coupe/2023/
16:45:33 [INFO] [8/140] https://www.drivearabia.com/carprices/uae/cadillac/cadillac-ats-coupe/2024/
16:45:34 [INFO] 

In [ ]:
"""
DriveArabia UAE isuzu Price Scraper
=====================================
Scrapes all isuzu models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/isuzu/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_isuzu.py
    python scrape_drivearabia_isuzu.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_isuzu.py --models isuzu-land-cruiser isuzu-camry
    python scrape_drivearabia_isuzu.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_isuzu.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
isuzu_INDEX = f"{BASE_URL}/carprices/uae/isuzu/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"isuzu|mitsubishi|jeep|dodge|isuzu|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|isuzu|"
    r"isuzu|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|isuzu)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": isuzu_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "isuzu-d-max",
    "isuzu-d-max-arctic-trucks-at35",
    "isuzu-mu-x",
    "isuzu-trooper"
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering isuzu models from %s", isuzu_INDEX)
    soup = fetch(session, isuzu_INDEX)
    if not soup:
        log.error("Failed to fetch isuzu index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/isuzu/(isuzu-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/isuzu/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/isuzu.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_isuzu import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["isuzu-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape isuzu vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/isuzu.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



16:48:42 [INFO] Discovering isuzu models from https://www.drivearabia.com/carprices/uae/isuzu/
16:48:42 [INFO] Discovered 4 models
16:48:42 [INFO] Year range: 2022â€“2026  |  Models: 4  |  Total pages: 20
16:48:43 [INFO] [1/20] https://www.drivearabia.com/carprices/uae/isuzu/isuzu-d-max/2022/
16:48:44 [INFO]   â†’ 6 trims
16:48:45 [INFO] [2/20] https://www.drivearabia.com/carprices/uae/isuzu/isuzu-d-max/2023/
16:48:46 [INFO] [3/20] https://www.drivearabia.com/carprices/uae/isuzu/isuzu-d-max/2024/
16:48:47 [INFO]   â†’ 6 trims
16:48:48 [INFO] [4/20] https://www.drivearabia.com/carprices/uae/isuzu/isuzu-d-max/2025/
16:48:49 [INFO]   â†’ 6 trims
16:48:50 [INFO] [5/20] https://www.drivearabia.com/carprices/uae/isuzu/isuzu-d-max/2026/
16:48:51 [INFO]   â†’ 6 trims
16:48:52 [INFO] [6/20] https://www.drivearabia.com/carprices/uae/isuzu/isuzu-d-max-arctic-trucks-at35/2022/
16:48:53 [INFO] [7/20] https://www.drivearabia.com/carprices/uae/isuzu/isuzu-d-max-arctic-trucks-at35/2023/
16:48:54 [INFO

: 

In [2]:
"""
DriveArabia UAE bentley Price Scraper
=====================================
Scrapes all bentley models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/bentley/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_bentley.py
    python scrape_drivearabia_bentley.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_bentley.py --models bentley-land-cruiser bentley-camry
    python scrape_drivearabia_bentley.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_bentley.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
bentley_INDEX = f"{BASE_URL}/carprices/uae/bentley/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"bentley|mitsubishi|jeep|dodge|bentley|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|bentley|"
    r"bentley|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|bentley)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": bentley_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "bentley-arnage",
    "bentley-azure",
    "bentley-bentayga",
    "bentley-brooklands",
    "bentley-continental-flying-spur",
    "bentley-continental-gt",
    "bentley-continental-gtc",
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering bentley models from %s", bentley_INDEX)
    soup = fetch(session, bentley_INDEX)
    if not soup:
        log.error("Failed to fetch bentley index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/bentley/(bentley-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/bentley/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/bentley.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_bentley import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["bentley-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape bentley vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/bentley.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



10:50:42 [INFO] Discovering bentley models from https://www.drivearabia.com/carprices/uae/bentley/
10:50:42 [INFO] Discovered 10 models
10:50:42 [INFO] Year range: 2022â€“2026  |  Models: 10  |  Total pages: 50
10:50:45 [INFO] [1/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-arnage/2022/
10:50:46 [INFO] [2/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-arnage/2023/
10:50:47 [INFO] [3/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-arnage/2024/
10:50:48 [INFO] [4/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-arnage/2025/
10:50:50 [INFO] [5/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-arnage/2026/
10:50:51 [INFO] [6/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-azure/2022/
10:50:52 [INFO] [7/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-azure/2023/
10:50:53 [INFO] [8/50] https://www.drivearabia.com/carprices/uae/bentley/bentley-azure/2024/
10:50:54 [INFO] [9/50] https://www.drive

In [3]:
"""
DriveArabia UAE jaguar Price Scraper
=====================================
Scrapes all jaguar models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/jaguar/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_jaguar.py
    python scrape_drivearabia_jaguar.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_jaguar.py --models jaguar-land-cruiser jaguar-camry
    python scrape_drivearabia_jaguar.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_jaguar.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
jaguar_INDEX = f"{BASE_URL}/carprices/uae/jaguar/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"jaguar|mitsubishi|jeep|dodge|jaguar|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|jaguar|"
    r"jaguar|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|jaguar)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": jaguar_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "jaguar-e-pace",
    "jaguar-f-pace",
    "jaguar-f-type",
    "jaguar-f-type-coupe",
    "jaguar-f-type-svr-convertible",
    "jaguar-f-type-svr-coupe",
    "jaguar-i-pace",
    "jaguar-s-type",
    "jaguar-x-type",
    "jaguar-xe",
    "jaguar-xf",
    "jaguar-xf-sportbrake",
    "jaguar-xfr-s",
    "jaguar-xj",
    "jaguar-xk",
    "jaguar-xk-convertible",
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering jaguar models from %s", jaguar_INDEX)
    soup = fetch(session, jaguar_INDEX)
    if not soup:
        log.error("Failed to fetch jaguar index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/jaguar/(jaguar-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/jaguar/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/jaguar.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_jaguar import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["jaguar-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape jaguar vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/jaguar.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



10:52:03 [INFO] Discovering jaguar models from https://www.drivearabia.com/carprices/uae/jaguar/
10:52:04 [INFO] Discovered 16 models
10:52:04 [INFO] Year range: 2022â€“2026  |  Models: 16  |  Total pages: 80
10:52:06 [INFO] [1/80] https://www.drivearabia.com/carprices/uae/jaguar/jaguar-e-pace/2022/
10:52:06 [INFO]   â†’ 2 trims
10:52:08 [INFO] [2/80] https://www.drivearabia.com/carprices/uae/jaguar/jaguar-e-pace/2023/
10:52:08 [INFO]   â†’ 2 trims
10:52:09 [INFO] [3/80] https://www.drivearabia.com/carprices/uae/jaguar/jaguar-e-pace/2024/
10:52:10 [INFO]   â†’ 1 trims
10:52:11 [INFO] [4/80] https://www.drivearabia.com/carprices/uae/jaguar/jaguar-e-pace/2025/
10:52:12 [INFO] [5/80] https://www.drivearabia.com/carprices/uae/jaguar/jaguar-e-pace/2026/
10:52:13 [INFO] [6/80] https://www.drivearabia.com/carprices/uae/jaguar/jaguar-f-pace/2022/
10:52:14 [INFO]   â†’ 10 trims
10:52:15 [INFO] [7/80] https://www.drivearabia.com/carprices/uae/jaguar/jaguar-f-pace/2023/
10:52:16 [INFO]   â†’ 10 t

In [4]:
"""
DriveArabia UAE baic Price Scraper
=====================================
Scrapes all baic models, their trims, and prices from:
  https://www.drivearabia.com/carprices/uae/baic/{model}/{year}/

Output CSV columns:
    model_slug, year, trim_name,
    price_min_aed, price_max_aed, price_avg_aed,
    price_raw, currency, url, scraped_at

Price logic:
    Range  (e.g. "AED 99,000 - AED 120,000") â†’ min + max + avg populated
    Single (e.g. "AED 245,000")               â†’ only avg populated

Usage:
    python scrape_drivearabia_baic.py
    python scrape_drivearabia_baic.py --start-year 2020 --end-year 2026
    python scrape_drivearabia_baic.py --models baic-land-cruiser baic-camry
    python scrape_drivearabia_baic.py --delay 2.0 --output my_prices.csv
    python scrape_drivearabia_baic.py --debug-html  # dumps raw HTML per page for inspection
"""

import re
import csv
import time
import logging
import argparse
from pathlib import Path
from datetime import UTC, datetime

import requests
from bs4 import BeautifulSoup, Tag

# â”€â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BASE_URL     = "https://www.drivearabia.com"
baic_INDEX = f"{BASE_URL}/carprices/uae/baic/"

DEFAULT_START_YEAR = 2022
DEFAULT_END_YEAR   = 2026
DEFAULT_DELAY      = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":  "document",
    "Sec-Fetch-Mode":  "navigate",
    "Sec-Fetch-Site":  "none",
    "Cache-Control":   "max-age=0",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# â”€â”€â”€ Known competitor makes (trim bleed filter) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_KNOWN_MAKES = re.compile(
    r"\b(hyundai|honda|nissan|ford|chevrolet|kia|mazda|bmw|mercedes|audi|"
    r"baic|mitsubishi|jeep|dodge|baic|infiniti|volkswagen|volvo|subaru|"
    r"genesis|porsche|land rover|jaguar|renault|peugeot|fiat|suzuki|baic|"
    r"baic|cadillac|buick|lincoln|acura|chrysler|skoda|seat|opel|citroen|"
    r"dacia|alfa romeo|maserati|ferrari|lamborghini|bentley|rolls royce|"
    r"aston martin|bugatti|mclaren|haval|geely|chery|mg|byd|great wall)\b",
    re.I,
)

# â”€â”€â”€ Trim name validator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_JUNK_WORDS = re.compile(
    r"\b(the|is|are|was|were|has|have|this|that|with|from|for|and|or|"
    r"view|compare|check|see|read|more|latest|new|used|buy|price|"
    r"review|photos|specs|variants|overview|starting|calculator|similar|"
    r"vehicle|brand|body|engine|fuel|weight|baic)\b",
    re.I,
)


def is_valid_trim(text: str) -> bool:
    if not text or len(text) > 70:
        return False
    if _KNOWN_MAKES.search(text):
        return False
    if re.search(r"(https?://|www\.)", text, re.I):
        return False
    if _JUNK_WORDS.search(text):
        return False
    # reject if >40% of chars are digits (price leaked into trim)
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False
    return True


# â”€â”€â”€ Price parsing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PRICE_NUM_RE = re.compile(r"\b(\d{1,3}(?:,\d{3})+|\d{4,8})\b")
_RANGE_SEP_RE = re.compile(r"\s*(?:–|—|â€“|â€”|-|to)\s*", re.I)
_AED_RE       = re.compile(r"AED\s*[\d,]+", re.I)


def _extract_nums(raw: str) -> list[int]:
    nums = []
    for m in _PRICE_NUM_RE.finditer(raw):
        digits = re.sub(r"[^\d]", "", m.group(1))
        if 4 <= len(digits) <= 8:
            nums.append(int(digits))
    return nums


def parse_prices(raw: str) -> dict:
    """
    Returns price_min_aed, price_max_aed, price_avg_aed.
    Range  â†’ all three populated.
    Single â†’ only price_avg_aed populated.
    """
    empty = {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": None}
    if not raw:
        return empty

    # Try range split
    halves = _RANGE_SEP_RE.split(raw, maxsplit=1)
    if len(halves) == 2:
        l = _extract_nums(halves[0])
        r = _extract_nums(halves[1])
        if l and r:
            lo, hi = min(l[0], r[0]), max(l[0], r[0])
            if lo != hi:
                return {
                    "price_min_aed": lo,
                    "price_max_aed": hi,
                    "price_avg_aed": round((lo + hi) / 2),
                }

    # Single price
    nums = _extract_nums(raw)
    if not nums:
        return empty
    return {"price_min_aed": None, "price_max_aed": None, "price_avg_aed": nums[0]}


# â”€â”€â”€ Session â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    # warm up cookies
    try:
        s.get(BASE_URL, timeout=10)
    except Exception:
        pass
    return s


def fetch(session: requests.Session, url: str, retries: int = 3) -> BeautifulSoup | None:
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15,
                               headers={"Referer": baic_INDEX})
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 404:
                log.debug("404 %s", url)
                return None
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning("Rate-limited. Sleeping %ds", wait)
                time.sleep(wait)
            else:
                log.warning("HTTP %d â€” %s (attempt %d)", resp.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Request error %s (attempt %d): %s", url, attempt, e)
            time.sleep(3 * attempt)
    return None


# â”€â”€â”€ HTML debug dump â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def dump_page_structure(soup: BeautifulSoup, slug: str, year: int) -> None:
    """
    Prints a structured summary of the page to help diagnose parser misses.
    Activate with --debug-html flag.
    """
    print(f"\n{'='*70}")
    print(f"DEBUG DUMP: {slug} / {year}")
    print(f"{'='*70}")

    tables = soup.find_all("table")
    print(f"\n[TABLES: {len(tables)}]")
    for i, t in enumerate(tables):
        print(f"  Table {i}:")
        for tr in t.find_all("tr")[:5]:
            cells = [td.get_text(strip=True)[:40] for td in tr.find_all(["td", "th"])]
            print(f"    {cells}")

    aed_nodes = soup.find_all(string=_AED_RE)
    print(f"\n[AED TEXT NODES: {len(aed_nodes)}]")
    for n in aed_nodes[:20]:
        p = n.parent
        gp = p.parent if p else None
        print(f"  text={repr(n.strip()[:60])}")
        print(f"    parent  â†’ <{p.name}> class={p.get('class')} text={p.get_text(strip=True)[:60]}")
        if gp:
            print(f"    grandp  â†’ <{gp.name}> class={gp.get('class')} text={gp.get_text(strip=True)[:60]}")

    price_els = soup.find_all(
        ["div", "span", "li", "td", "p"],
        class_=re.compile(r"price|trim|variant|spec|grade|version|car[-_]|row|item", re.I),
    )
    print(f"\n[PRICE/TRIM CLASS ELEMENTS: {len(price_els)}]")
    for el in price_els[:20]:
        print(f"  <{el.name}> class={el.get('class')} â†’ {el.get_text(strip=True)[:80]}")

    print(f"\n[ALL UL/OL LISTS (first 3)]")
    for ul in soup.find_all(["ul", "ol"])[:3]:
        for li in ul.find_all("li")[:6]:
            print(f"  <li> {li.get_text(strip=True)[:80]}")

    print(f"{'='*70}\n")


# â”€â”€â”€ Core parser: tries 5 strategies in order â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def parse_trims(
    soup: BeautifulSoup,
    model_slug: str,
    year: int,
    url: str,
    debug_html: bool = False,
) -> list[dict]:

    if debug_html:
        dump_page_structure(soup, model_slug, year)

    rows = []
    now  = datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")

    def make_row(trim_name: str, price_raw: str) -> dict:
        prices = parse_prices(price_raw)
        return {
            "model_slug":   model_slug,
            "year":         year,
            "trim_name":    trim_name.strip(),
            **prices,
            "price_raw":    price_raw.strip(),
            "currency":     "AED",
            "url":          url,
            "scraped_at":   now,
        }

    def normalize_lines(text: str) -> list[str]:
        return [
            re.sub(r"\s+", " ", ln).strip()
            for ln in text.splitlines()
            if re.sub(r"\s+", " ", ln).strip()
        ]

    def price_like(text: str) -> bool:
        if not _AED_RE.search(text):
            return False
        if re.search(r"\b(starting|calculator|monthly|similar|faq|highest|base price)\b", text, re.I):
            return False
        return bool(parse_prices(text)["price_avg_aed"])

    def add_unique(row_list: list[dict], trim_name: str, price_raw: str) -> None:
        if not is_valid_trim(trim_name) or not price_like(price_raw):
            return
        row = make_row(trim_name, price_raw)
        key = (row["trim_name"].lower(), row["price_raw"])
        existing = {
            (r["trim_name"].lower(), r["price_raw"])
            for r in row_list
        }
        if key not in existing:
            row_list.append(row)

    def parse_trim_price_lines(lines: list[str]) -> list[dict]:
        parsed = []
        i = 0
        while i < len(lines):
            line = lines[i]

            if i + 1 < len(lines) and is_valid_trim(line) and price_like(lines[i + 1]):
                add_unique(parsed, line, lines[i + 1])
                i += 2
                continue

            if price_like(line):
                m = _AED_RE.search(line)
                if m:
                    trim = line[:m.start()].strip(" :-")
                    price = line[m.start():].strip()
                    add_unique(parsed, trim, price)
            i += 1
        return parsed

    def section_after_heading(*heading_patterns: str) -> list[str]:
        headings = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
        for heading in headings:
            heading_text = heading.get_text(" ", strip=True)
            if not any(re.search(pattern, heading_text, re.I) for pattern in heading_patterns):
                continue

            section_lines = []
            for sib in heading.find_next_siblings():
                if not isinstance(sib, Tag):
                    continue
                if sib.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                    break
                text = sib.get_text("\n", strip=True)
                if text:
                    section_lines.extend(normalize_lines(text))
            return section_lines
        return []

    # S0: DriveArabia's real trim data is under "Trim Prices" or
    # "Original Trim Prices". Keep this scoped before broad AED scans so
    # unrelated prices do not leak in.
    section_rows = parse_trim_price_lines(
        section_after_heading(r"^(?:original\s+)?trim\s+prices$")
    )
    if section_rows:
        log.debug("S0 matched: %d rows", len(section_rows))
        return section_rows

    log.debug("No scoped Trim Prices section parsed for %s %s", model_slug, year)
    return []

    # â”€â”€ S1: standard <table> with â‰¥2 <td> per row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                trim  = tds[0].get_text(strip=True)
                price = tds[1].get_text(strip=True)
                # also try last cell if middle cells exist
                if len(tds) > 2 and not _AED_RE.search(price):
                    price = tds[-1].get_text(strip=True)
                if trim and (_AED_RE.search(price) or re.search(r"\d{5,}", price)):
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, price))
                    else:
                        log.debug("S1 rejected trim: %r", trim)
    if rows:
        log.debug("S1 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S2: <tr> where one <td> has AED and the prev sibling <td> has trim â”€â”€â”€
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        for i, td in enumerate(tds):
            cell_text = td.get_text(strip=True)
            if _AED_RE.search(cell_text):
                trim_td = tds[i - 1] if i > 0 else None
                if trim_td:
                    trim = trim_td.get_text(strip=True)
                    if is_valid_trim(trim):
                        rows.append(make_row(trim, cell_text))
    if rows:
        log.debug("S2 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S3: sibling element pairs â€” element with trim text next to price el â”€â”€
    #   DriveArabia uses various div/li layouts; look for any element whose
    #   text is AED-containing, then search previous siblings for trim text.
    price_els = soup.find_all(
        lambda tag: tag.name in ("div", "span", "li", "p", "td", "dd")
        and _AED_RE.search(tag.get_text())
        and len(tag.get_text(strip=True)) < 80
    )
    for el in price_els:
        price_text = el.get_text(strip=True)
        if not parse_prices(price_text)["price_avg_aed"]:
            continue
        trim = ""
        # search siblings first
        for sib in el.find_previous_siblings(limit=5):
            if not isinstance(sib, Tag):
                continue
            candidate = sib.get_text(strip=True)
            if is_valid_trim(candidate):
                trim = candidate
                break
        # then try parent's previous siblings
        if not trim and el.parent:
            for sib in el.parent.find_previous_siblings(limit=3):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
        rows.append(make_row(trim or "Unknown", price_text))
    if rows:
        log.debug("S3 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S4: scan all text nodes for AED, pull trim from nearest named ancestor
    for node in soup.find_all(string=_AED_RE):
        price_raw = node.strip()
        if len(price_raw) > 100:
            continue
        if not parse_prices(price_raw)["price_avg_aed"]:
            continue
        trim = ""
        # walk up the DOM tree looking for a sibling/cousin with trim text
        el = node.parent
        for _ in range(4):           # up to 4 levels up
            if el is None:
                break
            for sib in el.find_previous_siblings(limit=4):
                if not isinstance(sib, Tag):
                    continue
                candidate = sib.get_text(strip=True)
                if is_valid_trim(candidate):
                    trim = candidate
                    break
            if trim:
                break
            el = el.parent
        rows.append(make_row(trim or "Unknown", price_raw))
    if rows:
        log.debug("S4 matched: %d rows", len(rows))
        return rows

    # â”€â”€ S5: last resort â€” full page text line scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    #   Split all visible text into lines; look for consecutive lines where
    #   one looks like a trim and the next looks like a price.
    lines = [
        ln.strip()
        for ln in soup.get_text(separator="\n").splitlines()
        if ln.strip()
    ]
    i = 0
    while i < len(lines) - 1:
        if _AED_RE.search(lines[i + 1]) and is_valid_trim(lines[i]):
            rows.append(make_row(lines[i], lines[i + 1]))
            i += 2
            continue
        if _AED_RE.search(lines[i]) and is_valid_trim(lines[i - 1] if i > 0 else ""):
            rows.append(make_row(lines[i - 1], lines[i]))
            i += 2
            continue
        i += 1
    if rows:
        log.debug("S5 matched: %d rows", len(rows))

    return rows


# â”€â”€â”€ Model discovery â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

KNOWN_MODELS = [
    "baic-a1-hatchback",
    "baic-a1-sedan",
    "baic-a5-sedan",
    "baic-bj30",
    "baic-bj40",
    "baic-bj60",
    "baic-bj80",
    "baic-d20",
    "baic-u5-plus",
    "baic-x35",
    "baic-x55",
    "baic-x7",
]


def discover_models(session: requests.Session) -> list[str]:
    log.info("Discovering baic models from %s", baic_INDEX)
    soup = fetch(session, baic_INDEX)
    if not soup:
        log.error("Failed to fetch baic index page â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    slugs   = set()
    pattern = re.compile(r"^/carprices/uae/baic/(baic-[^/]+)/?$")
    for a in soup.find_all("a", href=True):
        m = pattern.match(a["href"])
        if m:
            slugs.add(m.group(1))

    if not slugs:
        log.warning("No slugs found via <a> â€” falling back to KNOWN_MODELS")
        return KNOWN_MODELS

    models = sorted(slugs)
    log.info("Discovered %d models", len(models))
    return models


# â”€â”€â”€ Scrape loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def scrape(
    models: list[str],
    years: list[int],
    delay: float,
    output_path: Path,
    debug_html: bool = False,
) -> None:
    session     = make_session()
    all_records = []
    total       = len(models) * len(years)
    done        = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for model in models:
        for year in years:
            url = f"{BASE_URL}/carprices/uae/baic/{model}/{year}/"
            done += 1
            log.info("[%d/%d] %s", done, total, url)

            soup = fetch(session, url)
            if soup is None:
                log.debug("Skipping %s %s (no response)", model, year)
                time.sleep(delay * 0.5)
                continue

            records = parse_trims(soup, model_slug=model, year=year,
                                  url=url, debug_html=debug_html)

            # defence-in-depth: drop rows where a competitor make leaked in
            before  = len(records)
            records = [r for r in records if not _KNOWN_MAKES.search(r["trim_name"])]
            if before - len(records):
                log.warning("  â†’ Dropped %d row(s) (competitor make in trim)", before - len(records))

            if records:
                log.info("  â†’ %d trims", len(records))
                all_records.extend(records)
            else:
                log.warning("  â†’ 0 trims parsed â€” try --debug-html to inspect page structure")

            time.sleep(delay)

    if not all_records:
        log.error("No records scraped. Not writing output file.")
        return

    fieldnames = [
        "model_slug", "year", "trim_name",
        "price_min_aed", "price_max_aed", "price_avg_aed",
        "price_raw", "currency", "url", "scraped_at",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    log.info("Wrote %d records â†’ %s", len(all_records), output_path)


# â”€â”€â”€ Public API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def main(
    start_year: int            = DEFAULT_START_YEAR,
    end_year:   int            = DEFAULT_END_YEAR,
    models:     list[str]|None = None,
    delay:      float          = DEFAULT_DELAY,
    output:     str            = "data/baic.csv",
    debug:      bool           = False,
    debug_html: bool           = False,
) -> None:
    """
    Jupyter usage:
        from scrape_drivearabia_baic import main
        main(start_year=2020, end_year=2026, delay=1.0)

        # To inspect raw page structure for one model:
        main(start_year=2024, end_year=2024, models=["baic-camry"], debug_html=True)
    """
    if debug or debug_html:
        logging.getLogger().setLevel(logging.DEBUG)

    if start_year > end_year:
        log.error("start_year (%d) must be <= end_year (%d)", start_year, end_year)
        return

    years   = list(range(start_year, end_year + 1))
    session = make_session()

    if models:
        resolved = models
        log.info("Using %d user-specified models", len(resolved))
    else:
        resolved = discover_models(session)
        if not resolved:
            log.error("Model discovery failed. Aborting.")
            return

    log.info("Year range: %dâ€“%d  |  Models: %d  |  Total pages: %d",
             start_year, end_year, len(resolved), len(resolved) * len(years))

    scrape(
        models=resolved,
        years=years,
        delay=delay,
        output_path=Path(output),
        debug_html=debug_html,
    )


# â”€â”€â”€ CLI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _cli_main() -> None:
    parser = argparse.ArgumentParser(
        description="Scrape baic vehicle prices from DriveArabia UAE",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--start-year", type=int, default=DEFAULT_START_YEAR,
                        help="First model year to scrape")
    parser.add_argument("--end-year",   type=int, default=DEFAULT_END_YEAR,
                        help="Last model year to scrape (inclusive)")
    parser.add_argument("--models",     nargs="+", default=None,
                        help="Specific model slugs (default: auto-discover)")
    parser.add_argument("--delay",      type=float, default=DEFAULT_DELAY,
                        help="Seconds between requests")
    parser.add_argument("--output",     default="data/baic.csv",
                        help="Output CSV path")
    parser.add_argument("--debug",      action="store_true",
                        help="Enable DEBUG logging")
    parser.add_argument("--debug-html", action="store_true",
                        help="Dump page structure to stdout (use with --models + 1 year to isolate)")
    args, _ = parser.parse_known_args()

    main(
        start_year=args.start_year,
        end_year=args.end_year,
        models=args.models,
        delay=args.delay,
        output=args.output,
        debug=args.debug,
        debug_html=args.debug_html,
    )


if __name__ == "__main__":
    _cli_main()



10:53:52 [INFO] Discovering baic models from https://www.drivearabia.com/carprices/uae/baic/
10:53:53 [INFO] Discovered 12 models
10:53:53 [INFO] Year range: 2022â€“2026  |  Models: 12  |  Total pages: 60
10:53:54 [INFO] [1/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-hatchback/2022/
10:53:56 [INFO] [2/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-hatchback/2023/
10:53:57 [INFO] [3/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-hatchback/2024/
10:53:58 [INFO] [4/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-hatchback/2025/
10:53:59 [INFO] [5/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-hatchback/2026/
10:54:00 [INFO] [6/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-sedan/2022/
10:54:01 [INFO] [7/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-sedan/2023/
10:54:02 [INFO] [8/60] https://www.drivearabia.com/carprices/uae/baic/baic-a1-sedan/2024/
10:54:04 [INFO] [9/60] https://www.drivearabia.com/carp